# Squares in Squares - Living Atlas of Tight Packings

Start from the attached **Fresh Records, Certified** notebook output, then merge the newest public packings puzzle by puzzle and replay this notebook's own certified improvements.

**Version 78 (06 November):** the attached public outputs complement each other, and the best single one (pramonoutomo's sis best packings 22 2498853780 kaggle verified, 22.2498701) is not the best packing for every `n`.
The merge also considers 0 stored packings (`n` = ) carried forward from earlier versions: local CUDA L-BFGS search, CPU contact-LP polishing, and external packings credited below. Each is retained only where it is the smallest certified packing. The combined result is **22.2498701551**, compared with 22.2498701471 for the best single attached notebook.

Each public packing is admitted only after the same float64 round trip and official certification; border expansion and deletion then run on the merged packings.
Coordinates are round-trip doubles, and centre dilation opens contacts by about `1e-11`. The final CSV is read with pandas' default float64 parser and scored exactly as on Kaggle.
Earlier versions replayed 60 GPU-search gains from 22 October; public packings have since beaten all of them, so their data was replaced by the version 7 improvements.

The default run replays compressed, locally found coordinates. The full search implementation is included below; `RECOMPUTE_SEARCH=True` reruns a bounded search and can take substantially longer.

In [ ]:
import os,sys,time,json,glob,base64,lzma,hashlib,shutil,subprocess
from pathlib import Path
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
ROOT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
RECOMPUTE_SEARCH=False
WORKERS=int(os.environ.get('SQ_CERT_WORKERS',str(min(os.cpu_count() or 2,16))))
import mpmath
if not hasattr(mpmath.mpf(1),'man'):
    deps=ROOT/'metric_deps'
    subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(deps),'mpmath==1.4.1'],check=True)
    sys.path.insert(0,str(deps))
    for name in list(sys.modules):
        if name=='mpmath' or name.startswith('mpmath.'):sys.modules.pop(name)
    import mpmath

## Official metric and certification library

Both are embedded verbatim from the certified base. All admission checks use the unchanged competition metric.

In [ ]:
%%writefile metric.py
"""Metric for the "Packing N Unit Squares into the Smallest Square" competition.

Participants submit packings of n unit squares (n=1..1000) inside a bounding
square.  Each square is specified by its center (x, y) and rotation angle.
Coordinates must be plain decimal literals, optionally in scientific notation
(e.g. ``2.70710678118654752440`` or ``1.5e-3``).  Symbolic / closed-form
expressions are not accepted -- submit their decimal expansion instead, at as
many digits as you like.

Scoring
-------
For each puzzle n the bounding side length L is measured
translation-invariantly as ``max(max_x - min_x, max_y - min_y)`` over all
corners.  The per-puzzle score is the excess over the area lower bound

    score(n) = L / sqrt(n) - 1

which is 0.0 exactly when the packing achieves the area lower bound (only
possible for perfect squares packed on the grid) and grows above 0 as the
bounding square wastes area.  **Lower is better**; total score is the sum
across n=1..N.

+-------------------+---------------------------------------------------------+
| Condition         | Score                                                   |
+===================+=========================================================+
| feasible packing  | L / sqrt(n) - 1     (>= 0 always)                       |
+-------------------+---------------------------------------------------------+
| infeasible/invalid| L_0 / sqrt(n) - 1   (grid ceiling — no worse than       |
|                   |                      submitting the trivial grid)       |
+-------------------+---------------------------------------------------------+

For reference: ``L_B = sqrt(n)`` is the area lower bound and
``L_0 = ceil(sqrt(n))`` is the trivial grid packing.  The best-known side
length plays no part -- beating a record just yields a smaller excess -- so
this metric reads no baseline values at all.

Feasibility is checked with interval arithmetic (mpmath) at escalating
precision to certify that no two squares overlap.

Kaggle entry point
------------------
``score(solution, submission, row_id_column_name)`` — called by the Kaggle
backend.  ``solution`` is a DataFrame loaded from ``solution.csv``; only its
``n`` column is read, to fix which puzzles are scored (and which absent ones
are charged the grid ceiling).  ``submission`` is the participant CSV with
columns ``n, square_id, x, y, angle``.

As a side effect it writes ``submission_details.csv``, one row per puzzle with
columns ``n`` and ``L``::

    n,L
    1,1.00000000000000000000000000000000000000000000000000
    2,2.00000000000000000000000000000000000000000000000000
    ...

``L`` is the measured bounding side for puzzle n -- not the per-puzzle score --
written with exactly 50 digits after the decimal point.  Those digits are
significant: L is measured in ``L_DPS``-digit mpmath arithmetic straight from
the submitted decimal strings, not from a double.  Puzzles with no accepted L
(absent from the submission, unparseable, failing column validation, or
rejected as infeasible) are written as zero.
"""

import contextlib
import csv
import io
import math
import multiprocessing
import re
from collections import defaultdict
from collections.abc import Callable, Sequence
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path

import mpmath
import pandas as pd

# ---------------------------------------------------------------------------
# interval
# ---------------------------------------------------------------------------

IVal = mpmath.ctx_iv.ivmpf

# Working precision for the (non-interval) measurement of the bounding side L,
# and the number of digits after the decimal point written for it in
# submission_details.csv.  The margin between the two keeps every written digit
# correct: the only rounding is in the trig of the corner computation, tens of
# digits below the last digit reported.
L_DPS = 60
L_DECIMALS = 50


@contextlib.contextmanager
def working_dps(dps: int):
    """Run a block with both mpmath contexts at ``dps`` digits, then restore them.

    ``mpmath.iv`` carries its own precision, independent of ``mpmath.mp``:
    setting only ``mpmath.mp.dps`` leaves every interval operation at the
    default 53 bits, which caps what a contact can be certified to at ~1e-13 no
    matter how far the escalation sequence climbs.  Both contexts are set here
    because the interval work is interleaved with plain ``mp`` arithmetic.

    >>> with working_dps(500):
    ...     width = mpmath.mpf(mpmath.iv.mpf('0.1').delta)
    >>> width < mpmath.mpf(10) ** -500
    True
    """
    old_iv_dps = mpmath.iv.dps
    old_mp_dps = mpmath.mp.dps
    try:
        mpmath.iv.dps = dps
        mpmath.mp.dps = dps
        yield
    finally:
        mpmath.iv.dps = old_iv_dps
        mpmath.mp.dps = old_mp_dps


def make_interval(value: str, dps: int = 50) -> IVal:
    with working_dps(dps):
        return mpmath.iv.mpf(value)


def iv_cos(x: IVal) -> IVal:
    return mpmath.iv.cos(x)


def iv_sin(x: IVal) -> IVal:
    return mpmath.iv.sin(x)


_ESCALATION_SEQUENCE = [50, 100, 200, 500]


def escalate(
    fn: Callable[[int], bool | None],
    max_dps: int = 500,
    start_dps: int = 50,
) -> bool | None:
    for dps in _ESCALATION_SEQUENCE:
        if dps < start_dps:
            continue
        if dps > max_dps:
            break
        result = fn(dps)
        if result is not None:
            return result
    return None


def iv_le(a: IVal, b: IVal) -> bool | None:
    """Certified less-or-equal: True if a <= b, False if a > b, None if undecided.

    Comparing two overlapping intervals is indeterminate.  mpmath 1.3 reports
    that as None, 1.4 raises ValueError; both mean "undecided" here.
    """
    try:
        return a <= b
    except ValueError:
        return None


def iv_min(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``min``: the tightest interval containing the true minimum.

    Comparison operators are useless for this -- two overlapping intervals have
    no determinate order -- so take it endpoint-wise instead.  min of the lower
    bounds is a lower bound on the true min, and min of the upper bounds is an
    upper bound on it, whichever element turns out to be smallest.
    """
    return mpmath.iv.mpf([min(v.a for v in values), min(v.b for v in values)])


def iv_max(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``max``. See :func:`iv_min`."""
    return mpmath.iv.mpf([max(v.a for v in values), max(v.b for v in values)])


# ---------------------------------------------------------------------------
# parser
# ---------------------------------------------------------------------------

# Bounds the cost of parsing one coordinate.  Far above any useful precision
# (L is reported to 50 decimals), so no honest submission comes near it.
_MAX_LITERAL_CHARS = 1000

_DECIMAL_RE = re.compile(r'^[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?$')


def _is_decimal(s: str) -> bool:
    return len(s) <= _MAX_LITERAL_CHARS and bool(_DECIMAL_RE.match(s.strip()))


def _check_decimal(s: str) -> str:
    """Return the stripped literal, or raise if it is not a plain decimal.

    >>> _check_decimal(" 2.70710678118654752440 ")
    '2.70710678118654752440'
    >>> _check_decimal("1.5e-3")
    '1.5e-3'
    >>> _check_decimal("1 + sqrt(2)")
    Traceback (most recent call last):
        ...
    ValueError: not a decimal number: '1 + sqrt(2)'
    """
    s = s.strip()
    if not _is_decimal(s):
        shown = s if len(s) <= 40 else s[:40] + '...'
        raise ValueError(f'not a decimal number: {shown!r}')
    return s


def parse_value(s: str, dps: int = 50) -> IVal:
    """Parse a decimal literal into an enclosing interval at ``dps`` digits."""
    return make_interval(_check_decimal(s), dps)


def parse_value_mp(s: str, dps: int = L_DPS) -> mpmath.mpf:
    """Parse a decimal literal into an ``mpf`` at ``dps`` digits.

    >>> mpmath.nstr(parse_value_mp("2.70710678118654752440"), 21, strip_zeros=False)
    '2.70710678118654752440'
    """
    s = _check_decimal(s)
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        return mpmath.mpf(s)
    finally:
        mpmath.mp.dps = old_dps


def parse_value_float(s: str) -> float:
    """Parse a decimal literal to float.

    >>> parse_value_float("3.5")
    3.5
    >>> parse_value_float("-2.5")
    -2.5
    >>> parse_value_float("1.5e-3")
    0.0015

    Values too large for a double are rejected rather than becoming ``inf``,
    which would poison every downstream geometry check:

    >>> parse_value_float("1e400")
    Traceback (most recent call last):
        ...
    ValueError: value out of range for a double: '1e400'
    """
    s = _check_decimal(s)
    value = float(s)
    if not math.isfinite(value):
        raise ValueError(f'value out of range for a double: {s!r}')
    return value


# ---------------------------------------------------------------------------
# scoring
# ---------------------------------------------------------------------------


def score_puzzle(n: int, s_certified: float) -> float:
    """Score a single puzzle: L / sqrt(n) - 1 (lower is better).

    Result is >= 0 for any feasible packing (equality only for perfect-square
    grid packings).  The best-known side length s* plays no part: beating a
    record simply yields a smaller excess.

    Perfect-square grid packing scores 0 exactly:
    >>> score_puzzle(4, 2.0)
    0.0

    Larger L → larger (worse) score:
    >>> round(score_puzzle(5, 2.707), 6)
    0.210607

    Doubling L takes the score from 0 to 1:
    >>> score_puzzle(4, 4.0)
    1.0
    """
    L_B = math.sqrt(n)
    return s_certified / L_B - 1.0


def infeasible_score(n: int) -> float:
    """Penalty score for an infeasible / invalid submission for puzzle n.

    Equal to submitting the trivial grid packing: L_0 / sqrt(n) - 1.
    """
    return math.ceil(math.sqrt(n)) / math.sqrt(n) - 1.0


# ---------------------------------------------------------------------------
# geometry
# ---------------------------------------------------------------------------


@dataclass
class Square:
    x: float
    y: float
    angle: float
    x_str: str = ''
    y_str: str = ''
    angle_str: str = ''
    _corners_float: list[tuple[float, float]] = field(default_factory=list, repr=False)

    def corners_float(self) -> list[tuple[float, float]]:
        if not self._corners_float:
            self._corners_float = compute_corners_float(self.x, self.y, self.angle)
        return self._corners_float


def compute_corners_float(cx: float, cy: float, angle: float) -> list[tuple[float, float]]:
    """Return the four corners of a unit square centered at (cx, cy) rotated by angle.

    >>> corners = compute_corners_float(0.5, 0.5, 0.0)
    >>> [(round(x, 1), round(y, 1)) for x, y in corners]
    [(1.0, 1.0), (0.0, 1.0), (0.0, 0.0), (1.0, 0.0)]
    """
    c = math.cos(angle)
    s = math.sin(angle)
    h = 0.5
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def compute_corners_mp(cx: mpmath.mpf, cy: mpmath.mpf, angle: mpmath.mpf) -> list[tuple[mpmath.mpf, mpmath.mpf]]:
    """``compute_corners_float`` at the ambient ``mpmath.mp`` precision."""
    c = mpmath.cos(angle)
    s = mpmath.sin(angle)
    h = mpmath.mpf(1) / 2
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def measure_side_mp(squares: list['Square'], dps: int = L_DPS) -> mpmath.mpf:
    """Bounding side ``max(max_x - min_x, max_y - min_y)`` at ``dps`` digits.

    Measured from the submitted decimal strings, so a packing whose coordinates
    carry more than double precision keeps that precision in the reported L.

    >>> unit = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> measure_side_mp([unit])
    mpf('1.0')
    """
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        xs: list[mpmath.mpf] = []
        ys: list[mpmath.mpf] = []
        for sq in squares:
            corners = compute_corners_mp(
                parse_value_mp(sq.x_str, dps),
                parse_value_mp(sq.y_str, dps),
                parse_value_mp(sq.angle_str, dps),
            )
            for x, y in corners:
                xs.append(x)
                ys.append(y)
        return max(max(xs) - min(xs), max(ys) - min(ys))
    finally:
        mpmath.mp.dps = old_dps


def compute_corners_interval(cx_iv: IVal, cy_iv: IVal, angle_iv: IVal) -> list[tuple[IVal, IVal]]:
    c = iv_cos(angle_iv)
    s = iv_sin(angle_iv)
    h = mpmath.iv.mpf('0.5')
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx_iv + dx1, cy_iv + dy1),
        (cx_iv + dx2, cy_iv + dy2),
        (cx_iv - dx1, cy_iv - dy1),
        (cx_iv - dx2, cy_iv - dy2),
    ]


def compute_aabb(corners: list[tuple[float, float]]) -> tuple[float, float, float, float]:
    xs = [c[0] for c in corners]
    ys = [c[1] for c in corners]
    return (min(xs), min(ys), max(xs), max(ys))


def aabb_overlap(
    a: tuple[float, float, float, float],
    b: tuple[float, float, float, float],
) -> bool:
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def _project_corners_float(corners: list[tuple[float, float]], ax: float, ay: float) -> tuple[float, float]:
    projs = [ax * cx + ay * cy for cx, cy in corners]
    return (min(projs), max(projs))


def sat_check_float(sq1: Square, sq2: Square, undecided_tol: float = 1e-8) -> str:
    """Check overlap between two unit squares using the Separating Axis Theorem.

    Returns ``"separated"``, ``"overlapping"``, or ``"undecided"``.

    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(5.0, 5.0, 0.0))
    'separated'
    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0))
    'overlapping'
    """
    c1 = sq1.corners_float()
    c2 = sq2.corners_float()

    axes = _get_axes_float(sq1.angle, sq2.angle)

    min_overlap = float('inf')
    for ax, ay in axes:
        min1, max1 = _project_corners_float(c1, ax, ay)
        min2, max2 = _project_corners_float(c2, ax, ay)
        overlap = min(max1, max2) - max(min1, min2)
        if overlap < -undecided_tol:
            return 'separated'
        min_overlap = min(min_overlap, overlap)

    if min_overlap > undecided_tol:
        return 'overlapping'
    return 'undecided'


def _get_axes_float(angle1: float, angle2: float) -> list[tuple[float, float]]:
    c1, s1 = math.cos(angle1), math.sin(angle1)
    c2, s2 = math.cos(angle2), math.sin(angle2)
    return [(c1, s1), (-s1, c1), (c2, s2), (-s2, c2)]


def sat_check_interval(
    sq1_corners: list[tuple[IVal, IVal]],
    sq2_corners: list[tuple[IVal, IVal]],
    angle1_iv: IVal,
    angle2_iv: IVal,
) -> bool | None:
    c1_iv, s1_iv = iv_cos(angle1_iv), iv_sin(angle1_iv)
    c2_iv, s2_iv = iv_cos(angle2_iv), iv_sin(angle2_iv)
    axes = [(c1_iv, s1_iv), (-s1_iv, c1_iv), (c2_iv, s2_iv), (-s2_iv, c2_iv)]

    has_undecided = False
    for ax, ay in axes:
        projs1 = [ax * cx + ay * cy for cx, cy in sq1_corners]
        projs2 = [ax * cx + ay * cy for cx, cy in sq2_corners]

        # Endpoint-wise, via iv_min/iv_max: ordering two overlapping intervals
        # with < is indeterminate, and skipping the projections that compare
        # indeterminately would narrow the span -- which shrinks the overlap and
        # biases the test towards wrongly certifying a pair as separated.
        min1, max1 = iv_min(projs1), iv_max(projs1)
        min2, max2 = iv_min(projs2), iv_max(projs2)

        overlap = iv_min([max1, max2]) - iv_max([min1, min2])
        le_result = iv_le(overlap, mpmath.iv.mpf('0'))
        if le_result is True:
            return True
        if le_result is None:
            has_undecided = True

    if has_undecided:
        return None
    return False


def sort_and_sweep(squares: list[Square]) -> list[tuple[int, int]]:
    n = len(squares)
    if n <= 1:
        return []

    aabbs = []
    for i, sq in enumerate(squares):
        corners = sq.corners_float()
        aabb = compute_aabb(corners)
        aabbs.append((i, aabb))

    aabbs.sort(key=lambda x: x[1][0])

    pairs = []
    for i_idx in range(n):
        i, aabb_i = aabbs[i_idx]
        for j_idx in range(i_idx + 1, n):
            j, aabb_j = aabbs[j_idx]
            if aabb_j[0] > aabb_i[2]:
                break
            if aabb_overlap(aabb_i, aabb_j):
                pairs.append((i, j) if i < j else (j, i))

    return pairs


def verify_puzzle(
    squares: list[Square],
) -> tuple[bool, str]:
    """Check that no two squares overlap, using float SAT then interval arithmetic.

    >>> s1 = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> s2 = Square(1.5, 0.5, 0.0, x_str="1.5", y_str="0.5", angle_str="0")
    >>> verify_puzzle([s1, s2])
    (True, 'feasible')
    >>> verify_puzzle([Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0)])
    (False, 'Squares 0 and 1 overlap (float check)')
    """
    pairs = sort_and_sweep(squares)
    undecided_pairs = []
    for i, j in pairs:
        result = sat_check_float(squares[i], squares[j])
        if result == 'overlapping':
            return (False, f'Squares {i} and {j} overlap (float check)')
        if result == 'undecided':
            undecided_pairs.append((i, j))

    if undecided_pairs:

        def check_overlap_certified(dps: int) -> bool | None:
            # The whole pair loop runs inside the precision context: parsing at dps
            # digits is not enough, since every later interval operation widens its
            # result to whatever precision mpmath.iv is set to at the time.
            with working_dps(dps):
                still_undecided = []
                for i, j in undecided_pairs:
                    sq_i, sq_j = squares[i], squares[j]
                    cx_i = parse_value(sq_i.x_str, dps)
                    cy_i = parse_value(sq_i.y_str, dps)
                    ai = parse_value(sq_i.angle_str, dps)
                    cx_j = parse_value(sq_j.x_str, dps)
                    cy_j = parse_value(sq_j.y_str, dps)
                    aj = parse_value(sq_j.angle_str, dps)
                    corners_i = compute_corners_interval(cx_i, cy_i, ai)
                    corners_j = compute_corners_interval(cx_j, cy_j, aj)
                    result = sat_check_interval(corners_i, corners_j, ai, aj)
                    if result is False:
                        return False
                    if result is None:
                        still_undecided.append((i, j))
                if still_undecided:
                    return None
                return True

        result = escalate(check_overlap_certified)
        if result is False:
            return (False, 'Squares overlap (certified)')
        if result is None:
            return (False, 'Overlap undecidable at max precision')

    return (True, 'feasible')


# ---------------------------------------------------------------------------
# main / Kaggle entry point
# ---------------------------------------------------------------------------


class ParticipantVisibleError(Exception):
    pass


MAX_N = 1000
KAGGLE_CORES = 4
MIN_PUZZLES_FOR_MP = 4


def _parse_submission_csv(text: str) -> dict[int, list[dict]]:
    reader = csv.DictReader(io.StringIO(text))
    required_cols = {'n', 'square_id', 'x', 'y', 'angle'}
    if not required_cols.issubset(set(reader.fieldnames or [])):
        missing = required_cols - set(reader.fieldnames or [])
        raise ParticipantVisibleError(f'Missing columns: {missing}')

    puzzles: dict[int, list[dict]] = defaultdict(list)
    for row_num, row in enumerate(reader, start=2):
        try:
            n = int(row['n'])
        except (ValueError, TypeError):
            raise ParticipantVisibleError(f'Row {row_num}: invalid n value: {row["n"]!r}')
        if n < 1 or n > MAX_N:
            raise ParticipantVisibleError(f'Row {row_num}: n={n} out of range [1, {MAX_N}]')
        puzzles[n].append(row)

    return dict(puzzles)


def _validate_puzzle(n: int, rows: list[dict]) -> str | None:
    if len(rows) != n:
        return f'n={n}: expected {n} rows, got {len(rows)}'

    square_ids = set()
    for row in rows:
        try:
            sid = int(row['square_id'])
        except (ValueError, TypeError):
            return f'n={n}: invalid square_id: {row["square_id"]!r}'
        square_ids.add(sid)

    expected_ids = set(range(n))
    if square_ids != expected_ids:
        return f'n={n}: square_ids should be 0..{n - 1}, got {sorted(square_ids)}'

    return None


def _build_squares(n: int, rows: list[dict]) -> tuple[list[Square], mpmath.mpf]:
    rows_by_id = {int(r['square_id']): r for r in rows}

    squares = []
    for sid in range(n):
        row = rows_by_id[sid]
        x_str = row['x'].strip()
        y_str = row['y'].strip()
        angle_str = row['angle'].strip()

        try:
            x_f = parse_value_float(x_str)
            y_f = parse_value_float(y_str)
            angle_f = parse_value_float(angle_str)
        except ValueError as e:
            raise ParticipantVisibleError(f'n={n}, square {sid}: cannot parse coordinates: {e}')

        squares.append(
            Square(
                x=x_f,
                y=y_f,
                angle=angle_f,
                x_str=x_str,
                y_str=y_str,
                angle_str=angle_str,
            )
        )

    return squares, measure_side_mp(squares)


def evaluate_single_puzzle(n: int, rows: list[dict]) -> tuple[int, float, str | None, str]:
    """Evaluate one puzzle, returning ``(n, score, L, reason)``.

    ``L`` is decimal text with ``L_DECIMALS`` digits after the point, and is
    reported only for a packing that passed the feasibility check: an
    unparseable, malformed, or overlapping puzzle gets None, so no L is
    published for a packing the metric rejected.  Text rather than a float
    because a double cannot hold those digits; the score itself is a double,
    taken from the same high-precision measurement.
    """
    error = _validate_puzzle(n, rows)
    if error:
        return (n, infeasible_score(n), None, error)

    try:
        squares, s_mp = _build_squares(n, rows)
    except ParticipantVisibleError as e:
        return (n, infeasible_score(n), None, str(e))

    feasible, reason = verify_puzzle(squares)
    if not feasible:
        return (n, infeasible_score(n), None, f'infeasible: {reason}')

    puzzle_score = score_puzzle(n, float(s_mp))
    return (n, puzzle_score, _format_detail(s_mp), 'ok')


def _evaluate_puzzle_worker(args: tuple) -> tuple[int, float, str | None, str]:
    n, rows = args
    try:
        return evaluate_single_puzzle(n, rows)
    except Exception as e:  # noqa: BLE001 - one bad puzzle must not kill the pool
        return (n, infeasible_score(n), None, f'worker error: {e}')


def _evaluate_puzzles_parallel(
    puzzles: dict[int, list[dict]],
    puzzle_ns: set[int],
) -> tuple[float, list[tuple[int, float, str | None, str]]]:
    """Score every n in ``puzzle_ns``; rows for any other n are ignored."""
    work = [(n, rows) for n, rows in sorted(puzzles.items()) if n in puzzle_ns]

    if not work:
        results: list[tuple[int, float, str | None, str]] = []
    elif len(work) < MIN_PUZZLES_FOR_MP:
        results = [evaluate_single_puzzle(n, rows) for n, rows in work]
    else:
        with multiprocessing.Pool(processes=KAGGLE_CORES) as pool:
            results = pool.map(_evaluate_puzzle_worker, work)

    # Any scored n missing from the submission gets the grid-ceiling penalty
    # (lower-is-better metric — otherwise omitting a puzzle would improve the
    # total).
    seen = {n for n, _, _, _ in results}
    for n in puzzle_ns:
        if n not in seen:
            results.append((n, infeasible_score(n), None, 'missing from submission'))
    results.sort(key=lambda r: r[0])

    total = sum(puzzle_score for _, puzzle_score, _, _ in results)
    return total, results


def _format_detail(value: mpmath.mpf) -> str:
    """Decimal text for one per-puzzle L: exactly ``L_DECIMALS`` digits after the point.

    An ``mpf`` is a binary fraction (``man * 2**exp``), so it is converted
    exactly to a ``Fraction`` and rounded once, half-to-even, at the last
    written digit.  ``mpmath.nstr`` counts significant digits rather than
    decimal places, and ``format(mpf, '.50f')`` needs mpmath >= 1.4, so neither
    is usable here.

    >>> _format_detail(mpmath.mpf(2))[:12]
    '2.0000000000'
    >>> len(_format_detail(mpmath.mpf(2)).split('.')[1])
    50
    >>> _format_detail(parse_value_mp('1.41421356237309504880168872420969807856967187537695'))
    '1.41421356237309504880168872420969807856967187537695'
    """
    exact = Fraction(value.man) * Fraction(2) ** value.exp
    scale = 10**L_DECIMALS
    units = round(exact * scale)
    sign = '-' if units < 0 else ''
    whole, frac = divmod(abs(units), scale)
    return f'{sign}{whole}.{frac:0{L_DECIMALS}d}'


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
) -> float:
    """Kaggle entry point. Score a submission against the solution's puzzle set.

    ``solution`` is read for one thing only: the set of n to score.  Everything
    else about a puzzle -- best-known side length, proven-optimal flag, source
    -- is irrelevant to ``L / sqrt(n) - 1``.

    >>> import pandas as pd
    >>> sol = pd.DataFrame({"id": [0], "n": [1]})
    >>> sub = pd.DataFrame({"id": [0], "n": [1], "square_id": [0],
    ...     "x": ["0.5"], "y": ["0.5"], "angle": ["0"]})
    >>> score(sol, sub, "id")  # single unit square, L=1, L/sqrt(1) - 1 = 0.0
    0.0
    """
    solution = solution.drop(columns=[row_id_column_name])
    submission = submission.drop(columns=[row_id_column_name])

    puzzle_ns = {int(n) for n in solution['n']}

    text = submission.to_csv(index=False)
    puzzles = _parse_submission_csv(text)
    total, results = _evaluate_puzzles_parallel(puzzles, puzzle_ns)

    # L is the measured bounding side for puzzle n -- NOT the per-puzzle score
    # L/sqrt(n) - 1.  Puzzles with no accepted L (missing from the submission,
    # unparseable, failing column validation, or rejected as infeasible) keep
    # the historical 0 placeholder: an L measured off an overlapping packing is
    # not a real packing of n squares, and downstream harvesting treats every
    # published L as attainable.  Values arrive here already rendered as decimal
    # text with L_DECIMALS digits after the point; unlike a JSON float they are
    # not re-serialized downstream, so all 50 digits reach the stored file
    # verbatim.
    result_map = {n: L_text for n, _, L_text, _ in results if L_text is not None}
    zero = _format_detail(mpmath.mpf(0))

    output_dir = Path('/kaggle/working')
    if not output_dir.is_dir():
        output_dir = Path('.')
    with open(output_dir / 'submission_details.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'L'])
        for n in range(1, MAX_N + 1):
            writer.writerow([n, result_map.get(n, zero)])

    return float(total)

In [ ]:
%%writefile sq_merge.py
"""Per-n certified merge of public packings and exact literature certificates.

The leaderboard is a sum of 1000 independent puzzles, so each n simply takes the
smallest box among: public submissions, exact rational certificates (Evan
Daniel's format: x y t=tan(theta/2)), and two free consequences of any packing
of m squares in side s:
  * border expansion: j extra unit rows/columns hold m + j(2*floor(s) + j) squares in side s + j;
  * deletion: L_n <= L_m for every n < m.
Everything chosen is re-certified with the unchanged official metric before it is written.
"""
import csv, glob, math, os, time
from fractions import Fraction
from multiprocessing import Pool

import mpmath
import numpy as np
import pandas as pd

DPS = 70                # working digits for exact -> decimal conversion
# The scorer reads the CSV into float64 before measuring, so every gap must survive rounding to
# a double: coordinates are written as shortest round-trip doubles and contacts are opened by ~1e-11.
DILATE = Fraction(1, 10**11)   # centre dilation: exact contacts become certifiable gaps
GAP, SPACING = '1e-11', '1e-11'  # border squares; strings, parsed inside the high-precision context


# ---------------------------------------------------------------- sources

def load_submission(path):
    """{n: [(x, y, angle) as strings]} for every complete puzzle in a submission CSV."""
    df = pd.read_csv(path, dtype=str)
    out = {}
    for n, g in df.groupby(df.n.astype(int)):
        g = g.assign(sid=g.square_id.astype(int)).sort_values('sid')
        if len(g) == n and list(g.sid) == list(range(n)):
            out[n] = list(zip(g.x.str.strip(), g.y.str.strip(), g.angle.str.strip()))
    return out


def cert_header(path):
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip() and not line.lstrip().startswith('#'):
                n, S = line.split()[:2]
                return int(n), Fraction(S)


def cert_rows(path):
    """Certificate -> double strings, centres dilated by 1+DILATE about the box centre."""
    lines = [l.split() for l in open(path, encoding='utf-8').read().splitlines()
             if l.strip() and not l.lstrip().startswith('#')]
    n, S = int(lines[0][0]), Fraction(lines[0][1])
    sq = [tuple(Fraction(v) for v in l[:3]) for l in lines[1:]]
    assert len(sq) == n, (path, n, len(sq))
    c, k = S / 2, 1 + DILATE
    rows = []
    with mpmath.workdps(DPS):
        for x, y, t in sq:
            x, y = c + (x - c) * k, c + (y - c) * k
            vals = (mpmath.mpf(x.numerator) / x.denominator, mpmath.mpf(y.numerator) / y.denominator,
                    2 * mpmath.atan(mpmath.mpf(t.numerator) / t.denominator))
            rows.append(tuple(repr(float(v)) for v in vals))
    return rows


# ---------------------------------------------------------------- geometry

def float_side(rows):
    a = np.array(rows, dtype=float)
    h = 0.5 * (np.abs(np.cos(a[:, 2])) + np.abs(np.sin(a[:, 2])))
    return max((a[:, 0] + h).max() - (a[:, 0] - h).min(), (a[:, 1] + h).max() - (a[:, 1] - h).min())


def expand(rows, j):
    """Add j unit columns on the right and j unit rows on top, all strictly separated."""
    with mpmath.workdps(DPS):
        gap, step, half = mpmath.mpf(GAP), 1 + mpmath.mpf(SPACING), mpmath.mpf(1) / 2
        xs, ys = [], []
        for x, y, a in rows:
            x, y, a = mpmath.mpf(x), mpmath.mpf(y), mpmath.mpf(a)
            h = (abs(mpmath.cos(a)) + abs(mpmath.sin(a))) / 2
            xs += [x - h, x + h]; ys += [y - h, y + h]
        x0, y0 = min(xs), min(ys)
        s = max(max(xs) - x0, max(ys) - y0)
        f = int(mpmath.floor(s - mpmath.mpf('1e-9')))
        new = []
        for i in range(j):
            new += [(x0 + s + gap + i * step + half, y0 + r * step + half) for r in range(f + j)]  # columns
            new += [(x0 + c * step + half, y0 + s + gap + i * step + half) for c in range(f)]      # rows
        return list(rows) + [(repr(float(x)), repr(float(y)), '0.0') for x, y in new]


def cert_floats(path):
    """Fast float view of a certificate (for searching; exact strings come from cert_rows)."""
    lines = [l.split() for l in open(path, encoding='utf-8').read().splitlines()
             if l.strip() and not l.lstrip().startswith('#')]
    return np.array([(float(Fraction(x)), float(Fraction(y)), 2 * math.atan(float(Fraction(t))))
                     for x, y, t in (l[:3] for l in lines[1:])])


def _clean_slab(lo, hi, tol=1e-7):
    """Unit slab [c, c+1] that no square crosses, holding the fewest squares: (count, c, inside) or None."""
    best = None
    for c in np.unique(np.round(lo, 9)):
        cross = ((lo < c - tol) & (hi > c + tol)) | ((lo < c + 1 - tol) & (hi > c + 1 + tol))
        if cross.any():
            continue
        inside = (lo >= c - tol) & (hi <= c + 1 + tol)
        if best is None or inside.sum() < best[0]:
            best = (int(inside.sum()), c, inside)
    return best


def slab_cut(a):
    """Reverse border expansion: remove one clean row slab and one clean column slab, close the gaps.

    Result: fewer squares in side s - 1 (or None).  Exact contacts survive, so the
    caller dilates the centres before writing.
    """
    for axis in (1, 0):
        h = 0.5 * (np.abs(np.cos(a[:, 2])) + np.abs(np.sin(a[:, 2])))
        cut = _clean_slab(a[:, axis] - h, a[:, axis] + h)
        if cut is None:
            return None
        a = a[~cut[2]].copy()
        a[a[:, axis] > cut[1] + 0.5, axis] -= 1
    return a if len(a) else None


def float_rows(a, dilate=1e-11):
    c = a[:, :2].mean(0)
    b = a.copy(); b[:, :2] = c + (b[:, :2] - c) * (1 + dilate)
    return [(repr(float(x)), repr(float(y)), repr(float(t))) for x, y, t in b]


def drop_to(rows, n):
    """Keep n squares, removing those furthest inside first (any subset is valid)."""
    a = np.array(rows, dtype=float)
    h = 0.5 * (np.abs(np.cos(a[:, 2])) + np.abs(np.sin(a[:, 2])))
    lo_x, hi_x = (a[:, 0] - h).min(), (a[:, 0] + h).max()
    lo_y, hi_y = (a[:, 1] - h).min(), (a[:, 1] + h).max()
    edge = np.minimum.reduce([a[:, 0] - h - lo_x, hi_x - a[:, 0] - h, a[:, 1] - h - lo_y, hi_y - a[:, 1] - h])
    keep = np.sort(np.argsort(-edge, kind='stable')[:n])
    return [rows[i] for i in keep]


# ---------------------------------------------------------------- selection

def choose(cands, N=1000, tol=1e-13, n_max=1100, layers=6):
    """cands: {(n, source): L}  ->  {n: (L, n_source, source)}."""
    own = {}
    for (n, src), L in cands.items():
        if n not in own or L < own[n][0] - tol:
            own[n] = (L, n, src)
    for n in range(1, n_max + 1):  # ascending, so expansions chain (328 -> 365 -> 404)
        if n not in own:
            continue
        L, _, src = own[n]
        f = math.floor(L - 1e-9)
        for j in range(1, layers + 1):
            n2 = n + j * (2 * f + j)
            if n2 > n_max:
                break
            if n2 not in own or L + j < own[n2][0] - tol:
                own[n2] = (L + j, n2, ('expand', n, src, j))
    pick, run = {}, (math.inf, None, None)
    for n in range(max(own), 0, -1):
        if n in own and own[n][0] < run[0]:
            run = own[n]
        if n <= N:
            pick[n] = own[n] if n in own and own[n][0] <= run[0] + tol else run
    return pick


def build(key, m, subs):
    if key[0] == 'expand':
        return expand(build(key[2], key[1], subs), key[3])
    if key[0] == 'cert':
        return cert_rows(key[1])
    if key[0] == 'cut':
        return float_rows(CUTS[key])
    return subs[key[1]][m]


def describe(key, root=''):
    if key[0] == 'expand':
        return f'{describe(key[2], root)}:{key[1]}+border{key[3]}'
    tail = f' x{key[2]}' if key[0] == 'cut' else ''
    return key[0] + ':' + (os.path.relpath(key[1], root).replace(os.sep, '/') if root else key[1]) + tail


# ---------------------------------------------------------------- certification with the official metric

def _certify(args):
    import metric
    n, rows = args
    recs = [{'n': str(n), 'square_id': str(i), 'x': x, 'y': y, 'angle': a} for i, (x, y, a) in enumerate(rows)]
    _, s, L, reason = metric.evaluate_single_puzzle(n, recs)
    return n, s, L, reason


def as_scored(rows):
    """What Kaggle's scorer sees: each value parsed to float64 and written back."""
    return [tuple(repr(float(v)) for v in r) for r in rows]


def certify(sol, workers=4):
    with Pool(workers) as p:
        return {n: (s, L, r) for n, s, L, r in p.imap_unordered(_certify, sorted(sol.items()), chunksize=4)}


def write_submission(sol, path):
    with open(path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['id', 'n', 'square_id', 'x', 'y', 'angle'])
        i = 0
        for n in sorted(sol):
            for sid, (x, y, a) in enumerate(sol[n]):
                w.writerow([i, n, sid, x, y, a]); i += 1


# ---------------------------------------------------------------- driver

CUTS = {}


def merge_all(sub_paths, cert_dir, out_dir, workers=4, log=print):
    t0 = time.time()
    subs = {p: load_submission(p) for p in sub_paths}
    cands = {(n, ('sub', p)): float_side(rows) for p, sub in subs.items() for n, rows in sub.items()}
    for path in glob.glob(os.path.join(cert_dir, '**', 'n*.cert'), recursive=True):
        if os.sep + 'parents' + os.sep in path or '/parents/' in path:
            continue
        n, S = cert_header(path)
        if n <= 1100:
            cands[(n, ('cert', path))] = float(S)
            a = cert_floats(path)
            for depth in (1, 2, 3):  # slab cuts of the certificate, repeated
                a = slab_cut(a)
                if a is None:
                    break
                CUTS[('cut', path, depth)] = a
                cands[(len(a), ('cut', path, depth))] = float_side(a)
    log(f'candidates ready ({time.time() - t0:.0f}s)')
    pick = choose(cands)
    sol, src = {}, {}
    for n, (L, m, key) in pick.items():
        rows = build(key, m, subs)
        # Source files can rely on contacts separated by less than one float64
        # ulp. Rewriting their decimals changes pandas' default CSV rounding.
        # Open a geometric margin before certification and serialization.
        sol[n] = float_rows(np.array(drop_to(rows, n) if m != n else rows, dtype=float), dilate=1e-11)
        src[n] = describe(key) + (f' minus {m - n}' if m != n else '')
    log(f'selected {len(sol)} puzzles from {len(subs)} submissions and {sum(k[1][0] == "cert" for k in cands)} certificates '
        f'({time.time() - t0:.0f}s)')

    res = certify(sol, workers)
    bad = sorted(n for n, (s, L, r) in res.items() if r != 'ok')
    log(f'not certified: {len(bad)} {bad[:20]}')
    if bad:  # fall back to the best input submission that certifies
        for n in bad:
            p = min((p for p in subs if n in subs[p]), key=lambda p: float_side(subs[p][n]))
            sol[n], src[n] = as_scored(subs[p][n]), ('sub', p, 'fallback')
        res.update(certify({n: sol[n] for n in bad}, workers))
    os.makedirs(out_dir, exist_ok=True)
    write_submission(sol, os.path.join(out_dir, 'submission.csv'))
    table = pd.DataFrame([(n, str(src[n]), res[n][1], res[n][0], res[n][2]) for n in sorted(sol)],
                         columns=['n', 'source', 'L', 'score', 'status'])
    table.to_csv(os.path.join(out_dir, 'submission_sources.csv'), index=False)
    log(f'certified total {table.score.sum()!r} ({time.time() - t0:.0f}s)')
    return table

## Load the attached base

The parent notebook is the only packing input. Its own credits and source provenance remain applicable.

In [ ]:
from sq_merge import load_submission,certify,write_submission,float_rows,float_side
inputs=Path(os.environ.get('SQ_INPUT','/kaggle/input'))
paths=list(inputs.rglob('submission.csv'))
assert paths,'Attach malyshevdanil/squares-in-squares-fresh-records-certified'
preferred=[p for p in paths if 'fresh-records-certified' in str(p)]
base_path=sorted(preferred or paths)[0]
base=load_submission(base_path)
assert sorted(base)==list(range(1,1001))
print('Base:',base_path)
base_checks=certify(base,workers=WORKERS)
assert all(r=='ok' for s,L,r in base_checks.values())
before=sum(base_checks[n][0] for n in sorted(base_checks))
assert abs(before-22.583444524031)<1e-9,(before,'Unexpected base version')
print(f'Base certified total: {before:.12f}')

## Reproducible search code

The extracted CUDA L-BFGS, SAT continuation and contact LP are adapted from lightningv08 (Apache-2.0). Seed selection, checkpointing and the local search schedule are provided here. Floating search candidates still require certification.

In [ ]:
Path('search').mkdir(exist_ok=True)

In [ ]:
%%writefile search/search_geometry.py
# Geometry adapted from lightningv08, Apache-2.0.
import math
import numpy as np
from scipy.spatial import cKDTree
GAP=1e-11
def side(p):
    h=(np.abs(np.cos(p[:,2]))+np.abs(np.sin(p[:,2])))/2
    lo=(p[:,:2]-h[:,None]).min(axis=0)
    hi=(p[:,:2]+h[:,None]).max(axis=0)
    return float(np.max(hi-lo))

def normalize(p, margin=GAP):
    p=np.asarray(p,dtype=float).copy()
    p[:,:2]*=1+margin
    h=(np.abs(np.cos(p[:,2]))+np.abs(np.sin(p[:,2])))/2
    p[:,:2]-=(p[:,:2]-h[:,None]).min(axis=0)
    return p

def grid(n):
    k=math.ceil(math.sqrt(n))
    return np.array([(i%k+.5,i//k+.5,0.) for i in range(n)])

def repair(p,clearance=4e-11,allow_compression=False):
    p=normalize(p,0);pairs=cKDTree(p[:,:2]).query_pairs(1.42,output_type='ndarray')
    if not len(pairs):return p
    i,j=pairs.T;d=p[j,:2]-p[i,:2];a=p[i,2];b=p[j,2]
    proj=np.maximum.reduce([abs(d[:,0]*np.cos(a)+d[:,1]*np.sin(a)),abs(-d[:,0]*np.sin(a)+d[:,1]*np.cos(a)),abs(d[:,0]*np.cos(b)+d[:,1]*np.sin(b)),abs(-d[:,0]*np.sin(b)+d[:,1]*np.cos(b))])
    rad=.5*(1+abs(np.cos(a-b))+abs(np.sin(a-b)))
    f=float(np.max((rad+clearance)/np.maximum(proj,1e-30)))
    if not allow_compression:f=max(1.,f)
    if not .9999<f<1.003:return None
    p[:,:2]*=f;return normalize(p,0)

In [ ]:
%%writefile search/contact_polish.py
# Adapted from lightningv08 (Apache-2.0); imports isolated for local search.
"""Trust-region sequential LP with free centers/angles and exact SAT admission.

Background: coupled sparse LP refinement in prvsiyan's A Square Deal notebook.
This vectorized constraint builder and experiment runner are implemented here.
"""
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1');os.environ.setdefault('OMP_NUM_THREADS','1')
import argparse,json,math,time
from pathlib import Path
import numpy as np
from scipy.optimize import linprog
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from search_geometry import side,normalize
from search_geometry import repair


def polish(p,rounds=28,seed=0,max_seconds=15):
    p=normalize(p,0);n=len(p);rng=np.random.default_rng(seed);at=time.perf_counter();step=.025;trace=[]
    for it in range(rounds):
        if time.perf_counter()-at>max_seconds:break
        L=side(p);theta=p[:,2];c=np.cos(theta);s=np.sin(theta);h=(abs(c)+abs(s))/2
        edges=cKDTree(p[:,:2]).query_pairs(1.75,output_type='ndarray');i,j=edges.T;d=p[j,:2]-p[i,:2]
        ux=np.column_stack((c[i],-s[i],c[j],-s[j]));uy=np.column_stack((s[i],c[i],s[j],c[j]));proj=d[:,0,None]*ux+d[:,1,None]*uy
        ranks=abs(proj);rank=ranks+rng.uniform(-2e-12,2e-12,ranks.shape);chosen=rank.argmax(1);rr=np.arange(len(i))
        sign=np.sign(proj[rr,chosen]);vx=sign*ux[rr,chosen];vy=sign*uy[rr,chosen];rotation=-d[:,0]*vy+d[:,1]*vx
        delta=theta[j]-theta[i];cd=np.cos(delta);sd=np.sin(delta);rad=(1+abs(cd)+abs(sd))/2;dr=(-sd*np.sign(cd)+cd*np.sign(sd))/2
        dist=abs(proj[rr,chosen]);own=chosen<2
        columns=np.column_stack((3*i,3*i+1,3*j,3*j+1,3*i+2,3*j+2))
        values=np.column_stack((vx,vy,-vx,-vy,-dr-np.where(own,rotation,0),dr-np.where(own,0,rotation)))
        rows=[];cols=[];vals=[];bounds=[];offset=0
        def add(cc,vv,bb):
            nonlocal offset
            bb=np.asarray(bb);count=len(bb)
            rows.append(np.repeat(np.arange(offset,offset+count),cc.shape[1]));cols.append(cc.ravel());vals.append(vv.ravel());bounds.append(bb);offset+=count
        add(columns,values,dist-rad-4e-11)
        # At relative right angles, both one-sided radius derivatives are needed.
        kink=np.flatnonzero(np.minimum(abs(cd),abs(sd))<1e-8)
        for slope in [-.5,.5]:
            vv=values[kink].copy();vv[:,4]=-slope-np.where(own[kink],rotation[kink],0);vv[:,5]=slope-np.where(own[kink],0,rotation[kink])
            add(columns[kink],vv,dist[kink]-rad[kink]-4e-11)
        k=np.arange(n);dh=(-s*np.sign(c)+c*np.sign(s))/2
        aligned=np.minimum(abs(c),abs(s))<1e-8
        for dim in [0,1]:
            for upper in [False,True]:
                ids=k;extra=k[aligned]
                for ids,deriv in [(ids,dh),(extra,np.full(len(extra),-.5)),(extra,np.full(len(extra),.5))]:
                    cc=np.column_stack((3*ids+dim,3*ids+2,np.full(len(ids),3*n)))
                    vv=np.column_stack((np.full(len(ids),1. if upper else -1.),deriv,np.full(len(ids),-1. if upper else 0.)))
                    bb=L-p[ids,dim]-h[ids] if upper else p[ids,dim]-h[ids]
                    add(cc,vv,bb)
        k=np.arange(3*n);u=3*n+1+k
        for sign in [-1.,1.]:add(np.column_stack((k,u)),np.tile([sign,-1.],(3*n,1)),np.zeros(3*n))
        A=coo_matrix((np.concatenate(vals),(np.concatenate(rows),np.concatenate(cols))),shape=(offset,6*n+1)).tocsr();rhs=np.concatenate(bounds)
        objective=np.zeros(6*n+1);objective[3*n]=1;objective[3*n+1:]=2e-6
        trust=np.tile([step,step,step*.4],n)
        limits=list(zip(-trust,trust))+[(-step,step)]+[(0,None)]*(3*n)
        fit=linprog(objective,A_ub=A,b_ub=rhs,bounds=limits,method='highs',options={'primal_feasibility_tolerance':1e-10,'dual_feasibility_tolerance':1e-9,'time_limit':2})
        if not fit.success:
            step*=.4
            if step<2e-7:break
            continue
        move=fit.x[:3*n].reshape(n,3);winner=None
        for alpha in [1.,.5,.25,.125,.0625,.03125]:
            trial=repair(p+alpha*move)
            if trial is not None and side(trial)<L-2e-10 and (winner is None or side(trial)<side(winner)):winner=trial
        if winner is None:
            step*=.4
            if step<2e-7:break
        else:
            p=winner;trace.append(dict(iteration=it,before=L,after=side(p),trust=step));step=min(.06,step*1.3)
    return p,trace,time.perf_counter()-at

def worker(job):
    n,p,seed,rounds=job;q,trace,seconds=polish(p,rounds,seed)
    return n,q,dict(n=n,seed=seed,seconds=seconds,before=side(p),after=side(q),trace=trace)

def run(input_path='artifacts/benchmark_packings.npz',tag='contact_polish',ns=None,rounds=28):
    from concurrent.futures import ProcessPoolExecutor
    z=np.load(input_path);bank={int(n):z[n] for n in z.files}
    if ns is None:ns=[500,502,504,595,596,597,645,646,647,648,750,751,858,924,414,696,855,626,810,998,328,545]
    jobs=[(n,bank[n],seed,rounds) for n in ns for seed in [0,29]];rows=[];started=time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as pool:
        for n,p,row in pool.map(worker,jobs):
            rows.append(row)
            if side(p)<side(bank[n]):bank[n]=p
            print(json.dumps({k:v for k,v in row.items() if k!='trace'}),flush=True)
            np.savez_compressed(f'artifacts/{tag}_packings.npz',**{str(n):p for n,p in bank.items()})
            Path(f'research/{tag}_status.json').write_text(json.dumps(dict(seconds=time.perf_counter()-started,method='Free-angle sequential sparse LP, kink-aware trust regions and exact nonlinear SAT admission',rows=rows),indent=2))
    export(f'artifacts/{tag}_submission.csv',bank,margin=4e-9)
    return bank,rows

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--input',default='artifacts/benchmark_packings.npz');p.add_argument('--tag',default='contact_polish');p.add_argument('--rounds',type=int,default=28);p.add_argument('--ns',type=int,nargs='*');a=p.parse_args();run(a.input,a.tag,a.ns,a.rounds)

In [ ]:
%%writefile search/cuda_backend.py
# From lightningv08 CUDA packing search, Apache-2.0.
import math
import numpy as np
import cupy as cp
CUDA=r'''
extern "C" {
__device__ double sg(double x){return (x>1e-12)-(x < -1e-12);}
__device__ double dot(const double*a,const double*b,int m,double*r){
 double z=0;for(int k=threadIdx.x;k<m;k+=blockDim.x)z+=a[k]*b[k];
 r[threadIdx.x]=z;__syncthreads();
 for(int t=blockDim.x/2;t;t/=2){if(threadIdx.x<t)r[threadIdx.x]+=r[threadIdx.x+t];__syncthreads();}return r[0];
}
__device__ double objective(double*q,double*g,int n,double L,int nc,int*head,int*next,double*cs,double*r,double*maxpen){
 int tid=threadIdx.x;
 for(int k=tid;k<nc*nc;k+=blockDim.x)head[k]=-1;
 __syncthreads();
 for(int i=tid;i<n;i+=blockDim.x){
  cs[2*i]=cos(q[3*i+2]);cs[2*i+1]=sin(q[3*i+2]);
  int x=max(0,min(nc-1,int(floor((q[3*i]+3)/1.5))));
  int y=max(0,min(nc-1,int(floor((q[3*i+1]+3)/1.5))));
  next[i]=atomicExch(head+y*nc+x,i);
 }__syncthreads();
 double energy=0,mp=0;
 for(int i=tid;i<n;i+=blockDim.x){
  double x=q[3*i],y=q[3*i+1],c=cs[2*i],s=cs[2*i+1];
  double gx=0,gy=0,ga=0,h=.5*(fabs(c)+fabs(s)),dh=.5*(-s*sg(c)+c*sg(s));
  double b[4]={h-x,h-y,x+h-L,y+h-L};
  for(int k=0;k<4;k++)if(b[k]>0){double p=b[k];energy+=2*p*p;mp=fmax(mp,p);ga+=4*p*dh;
   if(k==0)gx-=4*p;if(k==1)gy-=4*p;if(k==2)gx+=4*p;if(k==3)gy+=4*p;}
  int cx=max(0,min(nc-1,int(floor((x+3)/1.5)))),cy=max(0,min(nc-1,int(floor((y+3)/1.5))));
  for(int yy=max(0,cy-1);yy<=min(nc-1,cy+1);yy++)for(int xx=max(0,cx-1);xx<=min(nc-1,cx+1);xx++)
   for(int j=head[yy*nc+xx];j>=0;j=next[j])if(j!=i){
    double dx=q[3*j]-x,dy=q[3*j+1]-y;if(dx*dx+dy*dy>2.00000001)continue;
    double cj=cs[2*j],sj=cs[2*j+1],cd=c*cj+s*sj,sd=c*sj-s*cj;
    double rad=.5*(1+fabs(cd)+fabs(sd))+2e-10,rd=.5*(-sd*sg(cd)+cd*sg(sd));
    double ux[4]={c,-s,cj,-sj},uy[4]={s,c,sj,cj};int a=0;double z=dx*ux[0]+dy*uy[0];
    for(int k=1;k<4;k++){double v=dx*ux[k]+dy*uy[k];if(fabs(v)>fabs(z)){a=k;z=v;}}
    double p=rad-fabs(z);if(p<=0)continue;
    energy+=.25*p*p;mp=fmax(mp,p);double sign=sg(z);
    gx+=p*sign*ux[a];gy+=p*sign*uy[a];
    ga-=p*(rd+(a<2?sign*(-dx*uy[a]+dy*ux[a]):0));
   }
  g[3*i]=gx;g[3*i+1]=gy;g[3*i+2]=ga;
 }
 r[tid]=energy;r[blockDim.x+tid]=mp;__syncthreads();
 for(int t=blockDim.x/2;t;t/=2){if(tid<t){r[tid]+=r[tid+t];r[blockDim.x+tid]=fmax(r[blockDim.x+tid],r[blockDim.x+tid+t]);}__syncthreads();}
 if(tid==0)*maxpen=r[blockDim.x];__syncthreads();return r[0];
}
__global__ void eval(double*q,double*g,double*e,int n,int nc,double*Ls,int*head,int*next,double*cs){
 __shared__ double r[512],mp;
 int b=blockIdx.x;double v=objective(q+b*3*n,g+b*3*n,n,Ls[b],nc,head+b*nc*nc,next+b*n,cs+b*2*n,r,&mp);
 if(threadIdx.x==0){e[2*b]=v;e[2*b+1]=mp;}
}
__global__ void lbfgs(double*Q,double*work,double*history,double*out,int n,int nc,double*Ls,int*heads,int*links,double*trigs,int steps,int M){
 __shared__ double r[512],mp,alpha[12],rho[12],syarr[12],yyarr[12];
 int b=blockIdx.x,tid=threadIdx.x,d=3*n;
 double*q=Q+b*d,*g=work+b*d*5,*v=g+d,*trial=v+d,*gt=trial+d,*direction=gt+d;
 double*S=history+b*2*M*d,*Y=S+M*d;int*head=heads+b*nc*nc,*next=links+b*n;double*cs=trigs+b*2*n;
 double L=Ls[b],E=objective(q,g,n,L,nc,head,next,cs,r,&mp);int stored=0,last=0,its=0;
 for(int it=0;it<steps;it++){
  its=it;if(mp<2e-11)break;
  for(int k=tid;k<d;k+=blockDim.x)v[k]=g[k];__syncthreads();
  for(int j=0;j<stored;j++){int h=(last-1-j+M)%M;double z=dot(S+h*d,v,d,r)*rho[h];
   if(tid==0)alpha[h]=z;__syncthreads();for(int k=tid;k<d;k+=blockDim.x)v[k]-=z*Y[h*d+k];__syncthreads();}
  double scale=stored?fmin(10.,fmax(1e-6,syarr[(last-1+M)%M]/yyarr[(last-1+M)%M])):.2;
  for(int k=tid;k<d;k+=blockDim.x)v[k]*=scale;__syncthreads();
  for(int j=stored-1;j>=0;j--){int h=(last-1-j+M)%M;double z=dot(Y+h*d,v,d,r)*rho[h];
   double a=alpha[h]-z;for(int k=tid;k<d;k+=blockDim.x)v[k]+=a*S[h*d+k];__syncthreads();}
  for(int k=tid;k<d;k+=blockDim.x)direction[k]=-v[k];__syncthreads();
  double slope=dot(g,direction,d,r);
  if(slope>=-1e-30){for(int k=tid;k<d;k+=blockDim.x)direction[k]=-.2*g[k];__syncthreads();slope=dot(g,direction,d,r);stored=0;}
  double dm=0;for(int k=tid;k<d;k+=blockDim.x)dm=fmax(dm,fabs(direction[k]));r[tid]=dm;__syncthreads();
  for(int t=blockDim.x/2;t;t/=2){if(tid<t)r[tid]=fmax(r[tid],r[tid+t]);__syncthreads();}
  double step=fmin(1.,.2/fmax(1e-30,r[0])),En=E;bool ok=false;
  for(int ls=0;ls<24;ls++){
   for(int k=tid;k<d;k+=blockDim.x)trial[k]=q[k]+step*direction[k];__syncthreads();
   En=objective(trial,gt,n,L,nc,head,next,cs,r,&mp);
   if(En<=E+1e-4*step*slope){ok=true;break;}step*=.5;
  }
  if(!ok)break;
  for(int k=tid;k<d;k+=blockDim.x){v[k]=trial[k]-q[k];direction[k]=gt[k]-g[k];}__syncthreads();
  double sy=dot(v,direction,d,r),yy=dot(direction,direction,d,r);
  if(sy>1e-24&&yy>1e-30){for(int k=tid;k<d;k+=blockDim.x){S[last*d+k]=v[k];Y[last*d+k]=direction[k];}__syncthreads();
   if(tid==0){rho[last]=1/sy;syarr[last]=sy;yyarr[last]=yy;}__syncthreads();last=(last+1)%M;stored=min(M,stored+1);}
  for(int k=tid;k<d;k+=blockDim.x){q[k]=trial[k];g[k]=gt[k];}__syncthreads();E=En;
 }
 if(tid==0){out[3*b]=E;out[3*b+1]=mp;out[3*b+2]=its;}
}
}
'''

mod=cp.RawModule(code=CUDA,options=('--std=c++17',),name_expressions=['eval','lbfgs'])

def buffers(q,L):
    q=cp.asarray(q,dtype=cp.float64);B,n,_=q.shape;nc=int(math.ceil((max(L)+6)/1.5))+1
    return q,n,nc,cp.asarray(L,dtype=cp.float64),cp.empty((B,nc*nc),cp.int32),cp.empty((B,n),cp.int32),cp.empty((B,n,2),cp.float64)

def energy(q,L):
    q,n,nc,L,h,l,cs=buffers(q,L);g=cp.empty_like(q);e=cp.empty((len(q),2),cp.float64)
    mod.get_function('eval')((len(q),),(256,),(q,g,e,np.int32(n),np.int32(nc),L,h,l,cs))
    return cp.asnumpy(e),cp.asnumpy(g)

def quench(q,L,steps=1200):
    q,n,nc,L,h,l,cs=buffers(q,L);B=len(q);M=8
    work=cp.empty((B,5,n,3),cp.float64);hist=cp.empty((B,2,M,n,3),cp.float64);out=cp.empty((B,3),cp.float64)
    mod.get_function('lbfgs')((B,),(256,),(q,work,hist,out,np.int32(n),np.int32(nc),L,h,l,cs,np.int32(steps),np.int32(M)))
    return cp.asnumpy(q),cp.asnumpy(out)

def check():
    rng=np.random.default_rng(17);q=rng.uniform(.4,2.5,(1,5,3));q[:,:,2]*=.2
    e,g=energy(q,[3.]);errs=[]
    for i,k in [(0,0),(1,2),(3,1),(4,2)]:
        plus=q.copy();minus=q.copy();plus[0,i,k]+=1e-6;minus[0,i,k]-=1e-6
        fd=(energy(plus,[3.])[0][0,0]-energy(minus,[3.])[0][0,0])/2e-6
        errs.append(abs(fd-g[0,i,k]))
    assert max(errs)<1e-6,errs
    qq,o=quench(q,[3.],100)
    assert o[0,0]<e[0,0],(e,o)
    print('CUDA analytic gradient / descent checks passed',max(errs),flush=True)

In [ ]:
%%writefile search/cuda_precision.py
# From lightningv08 CUDA packing search, Apache-2.0.
import numpy as np
import cupy as cp
import cuda_backend as base
OBJECTIVE=r'''
__device__ double objective(double*q,double*g,int n,double L,int nc,int*head,int*next,double*cs,double*r,double*maxpen){
 int tid=threadIdx.x;const double tau=SMOOTH_TAU;
 for(int k=tid;k<nc*nc;k+=blockDim.x)head[k]=-1;__syncthreads();
 for(int i=tid;i<n;i+=blockDim.x){
  cs[2*i]=cos(q[3*i+2]);cs[2*i+1]=sin(q[3*i+2]);
  int x=max(0,min(nc-1,int(floor((q[3*i]+3)/1.5)))),y=max(0,min(nc-1,int(floor((q[3*i+1]+3)/1.5))));
  next[i]=atomicExch(head+y*nc+x,i);
 }__syncthreads();
 double energy=0,mp=0;
 for(int i=tid;i<n;i+=blockDim.x){
  double x=q[3*i],y=q[3*i+1],c=cs[2*i],s=cs[2*i+1];
  double ac=sqrt(c*c+tau*tau),as=sqrt(s*s+tau*tau),h=.5*(ac+as),dh=.5*(-s*c/ac+c*s/as);
  double gx=0,gy=0,ga=0,b[4]={h-x,h-y,x+h-L,y+h-L};
  for(int k=0;k<4;k++)if(b[k]>0){double p=b[k];energy+=2*p*p;mp=fmax(mp,p);ga+=4*p*dh;
   if(k==0)gx-=4*p;if(k==1)gy-=4*p;if(k==2)gx+=4*p;if(k==3)gy+=4*p;}
  int cx=max(0,min(nc-1,int(floor((x+3)/1.5)))),cy=max(0,min(nc-1,int(floor((y+3)/1.5))));
  for(int yy=max(0,cy-1);yy<=min(nc-1,cy+1);yy++)for(int xx=max(0,cx-1);xx<=min(nc-1,cx+1);xx++)
   for(int j=head[yy*nc+xx];j>=0;j=next[j])if(j!=i){
    double dx=q[3*j]-x,dy=q[3*j+1]-y;if(dx*dx+dy*dy>2.0000001)continue;
    double cj=cs[2*j],sj=cs[2*j+1],cd=c*cj+s*sj,sd=c*sj-s*cj;
    double rc=sqrt(cd*cd+tau*tau),rs=sqrt(sd*sd+tau*tau);
    double rad=.5*(1+rc+rs)+4e-11,rd=.5*(-sd*cd/rc+cd*sd/rs);
    double ux[4]={c,-s,cj,-sj},uy[4]={s,c,sj,cj},z[4],az[4],w[4],peak=-1e100;
    for(int k=0;k<4;k++){z[k]=dx*ux[k]+dy*uy[k];peak=fmax(peak,fabs(z[k]));}
    if(peak-2*tau>=rad)continue;
    for(int k=0;k<4;k++)az[k]=sqrt(z[k]*z[k]+tau*tau)-tau;
    double d01=sqrt((az[0]-az[1])*(az[0]-az[1])+tau*tau),d23=sqrt((az[2]-az[3])*(az[2]-az[3])+tau*tau);
    double m01=.5*(az[0]+az[1]+d01-tau),m23=.5*(az[2]+az[3]+d23-tau),dt=sqrt((m01-m23)*(m01-m23)+tau*tau);
    double top=.5*(1+(m01-m23)/dt),a01=.5*(1+(az[0]-az[1])/d01),a23=.5*(1+(az[2]-az[3])/d23);
    w[0]=top*a01;w[1]=top*(1-a01);w[2]=(1-top)*a23;w[3]=(1-top)*(1-a23);
    double projection=.5*(m01+m23+dt-tau),p=rad-projection;
    if(p<=0)continue;energy+=.25*p*p;mp=fmax(mp,p);
    double px=0,py=0,pa=0;
    for(int k=0;k<4;k++){double t=w[k]*z[k]/(az[k]+tau);px+=t*ux[k];py+=t*uy[k];if(k<2)pa+=t*(-dx*uy[k]+dy*ux[k]);}
    gx+=p*px;gy+=p*py;ga-=p*(rd+pa);
   }
  g[3*i]=gx;g[3*i+1]=gy;g[3*i+2]=ga;
 }
 r[tid]=energy;r[blockDim.x+tid]=mp;__syncthreads();
 for(int t=blockDim.x/2;t;t/=2){if(tid<t){r[tid]+=r[tid+t];r[blockDim.x+tid]=fmax(r[blockDim.x+tid],r[blockDim.x+tid+t]);}__syncthreads();}
 if(tid==0)*maxpen=r[blockDim.x];__syncthreads();return r[0];
}
'''

modules={}

def objective_source(kind):
    if kind!='log':return OBJECTIVE
    start=OBJECTIVE.index('double ux[4]');end=OBJECTIVE.index('if(p<=0)continue;',start)
    old=r'''double ux[4]={c,-s,cj,-sj},uy[4]={s,c,sj,cj},z[4],az[4],w[4],peak=-1e100;
    for(int k=0;k<4;k++){z[k]=dx*ux[k]+dy*uy[k];az[k]=sqrt(z[k]*z[k]+tau*tau);peak=fmax(peak,az[k]-tau);}
    double sum=0;for(int k=0;k<4;k++){w[k]=exp((az[k]-tau-peak)/tau);sum+=w[k];}
    double projection=peak+tau*(log(sum)-1.3862943611198906188),p=rad-projection;
    '''
    return (OBJECTIVE[:start]+old+OBJECTIVE[end:]).replace('w[k]*z[k]/(az[k]+tau)','w[k]/sum*z[k]/az[k]')

def module(tau,kind='quadratic'):
    key=(tau,kind)
    if key not in modules:
        start=base.CUDA.index('__device__ double objective(');end=base.CUDA.index('__global__ void eval(')
        code=f'#define SMOOTH_TAU {tau:.17g}\n'+base.CUDA[:start]+objective_source(kind)+base.CUDA[end:]
        code=code.replace('if(mp<2e-11)break;','if(mp<5e-12)break;')
        if kind!='log':
            code=code.replace('int stored=0,last=0,its=0;','int stored=0,last=0,its=0,flat=0;')
            code=code.replace('E=En;','if(E-En<1e-13*fmax(E,1e-28))flat++;else flat=0; E=En;if(flat>60)break;')
        modules[key]=cp.RawModule(code=code,options=('--std=c++17',),name_expressions=['eval','lbfgs'])
    return modules[key]

def quench(q,L,tau,steps,kind='quadratic'):
    original=base.mod
    try:base.mod=module(tau,kind);return base.quench(q,L,steps)
    finally:base.mod=original

def check():
    rng=np.random.default_rng(818);q=rng.uniform(.4,2.5,(1,7,3));q[:,:,2]*=.5;L=np.array([3.])
    original=base.mod;base.mod=module(1e-3)
    try:
        e,g=base.energy(q,L);errors=[]
        for i,k in [(0,0),(1,2),(3,1),(4,2)]:
            a=q.copy();b=q.copy();a[0,i,k]+=1e-6;b[0,i,k]-=1e-6
            fd=(base.energy(a,L)[0][0,0]-base.energy(b,L)[0][0,0])/2e-6
            errors.append(abs(fd-g[0,i,k]))
        assert max(errors)<2e-6,errors
        print('Smooth SAT gradient check',max(errors),flush=True)
    finally:base.mod=original

In [ ]:
%%writefile search/run_cpu.py
"""Resumable contact LP search, adapted from lightningv08 (Apache-2.0).

Only writes inside nb2. A floating candidate is not a certified improvement.
"""
import os
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):
    os.environ[key]='1'
import sys,time,json,argparse
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor,as_completed
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
HERE=Path(__file__).resolve().parent
sys.path.insert(0,str(HERE.parent))
from sq_merge import load_submission
from contact_polish import polish
from search_geometry import side,normalize

PRIORITY=[803,860,455,501,919,748,547,599,647,859,918]

def worker(job):
    key,n,p,method,seconds=job
    start=time.time()
    q,trace,_=polish(p,rounds=80,seed=n,max_seconds=seconds)
    return key,n,q,dict(key=key,n=n,method=method,before=side(p),after=side(q),seconds=time.time()-start,steps=len(trace))

def jobs(bank,mode,seconds):
    table=pd.read_csv(HERE.parent/'out_merge6/submission_sources.csv')
    priority=list(dict.fromkeys(n+d for n in PRIORITY for d in (0,-1,1,-2,2,-3,3)))
    derived=list(table.loc[table.source.str.contains('border|minus'),'n'])
    order=list(dict.fromkeys(priority+derived+list(range(2,1001))))
    for n in order:
        if n<2 or n>1000 or side(bank[n])<=np.sqrt(n)+1e-8:continue
        if mode=='own':
            yield f'own_{n}',n,bank[n],f'contact LP own n={n}',seconds
        else:
            if n==1000:continue
            a=normalize(bank[n+1],0)
            # Boundary extrema plus squares having the most free nearest-neighbour distance.
            h=(abs(np.cos(a[:,2]))+abs(np.sin(a[:,2])))/2
            boundary=[]
            for k in (0,1):
                boundary.extend(np.argsort(a[:,k]-h)[:2]);boundary.extend(np.argsort(-(a[:,k]+h))[:2])
            d=cKDTree(a[:,:2]).query(a[:,:2],k=min(5,len(a)))[0]
            free=np.argsort(-d[:,1:].mean(axis=1))[:4]
            ids=list(dict.fromkeys(list(free)+boundary))
            if side(a)>side(bank[n])+.045:continue
            for i in ids:
                yield f'drop_{n}_{i}',n,np.delete(a,i,0),f'neighbour {n+1} remove {i} + contact LP',seconds

def main():
    ap=argparse.ArgumentParser();ap.add_argument('--mode',default='own');ap.add_argument('--seconds',type=float,default=12);ap.add_argument('--workers',type=int,default=6);ap.add_argument('--limit',type=int,default=1000);args=ap.parse_args()
    dest=HERE/'cpu_candidates';dest.mkdir(exist_ok=True)
    bank={n:np.array(r,float) for n,r in load_submission(HERE.parent/'out_merge6/submission.csv').items()}
    donepath=HERE/f'cpu_{args.mode}_done.json';done=set(json.loads(donepath.read_text())) if donepath.exists() else set()
    work=[j for j in jobs(bank,args.mode,args.seconds) if j[0] not in done][:args.limit]
    print(json.dumps(dict(event='start',mode=args.mode,jobs=len(work),workers=args.workers)),flush=True)
    with ProcessPoolExecutor(max_workers=args.workers) as pool:
        futures=[pool.submit(worker,j) for j in work]
        for future in as_completed(futures):
            key,n,q,row=future.result();row['gain']=(side(bank[n])-side(q))/np.sqrt(n)
            if row['gain']>1e-11:
                path=dest/f'{key}.npz';np.savez_compressed(path,p=q)
                path.with_suffix('.json').write_text(json.dumps(row,indent=2))
            print(json.dumps(row),flush=True)
            done.add(key);donepath.write_text(json.dumps(sorted(done)))
    print('COMPLETE',flush=True)

if __name__=='__main__':main()

In [ ]:
%%writefile search/run_gpu.py
"""Neighbour transfer and local squeeze on CUDA, with resumable checkpoints.

CUDA L-BFGS and conservative smooth SAT are credited to lightningv08,
Apache-2.0. Seed selection and this search schedule are new here.
"""
import os
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):
    os.environ[key]='1'
from pathlib import Path
HERE=Path(__file__).resolve().parent
os.environ['CUPY_CACHE_DIR']=str(HERE/'cupy_cache')
import sys,time,json,argparse
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
sys.path.insert(0,str(HERE.parent))
from sq_merge import load_submission
from search_geometry import side,normalize,repair
from contact_polish import polish
from run_cpu import PRIORITY

def insert_holes(p,rng,count=6):
    p=normalize(p,0);L=side(p)
    positions=rng.uniform(.5,L-.5,(1024,2))
    # Include box corners and a regular lattice of probes.
    grid=np.linspace(.5,L-.5,int(L*2)+1)
    xy=np.array(np.meshgrid(grid,grid)).reshape(2,-1).T
    positions=np.vstack((positions,xy))
    nearest=cKDTree(p[:,:2]).query(positions)[0]
    positions=positions[np.argsort(-nearest)[:128]]
    choices=[]
    for angle in (0.,np.pi/4,float(p[rng.integers(len(p)),2])):
        d=positions[:,None,:]-p[None,:,:2]
        c=np.cos(p[:,2]);s=np.sin(p[:,2]);ca=np.cos(angle);sa=np.sin(angle)
        proj=np.maximum.reduce([abs(d[:,:,0]*c+d[:,:,1]*s),abs(-d[:,:,0]*s+d[:,:,1]*c),abs(d[:,:,0]*ca+d[:,:,1]*sa),abs(-d[:,:,0]*sa+d[:,:,1]*ca)])
        rad=(1+abs(np.cos(p[:,2]-angle))+abs(np.sin(p[:,2]-angle)))/2
        gap=(proj-rad).min(axis=1)
        h=(abs(ca)+abs(sa))/2
        gap=np.minimum(gap,np.minimum(positions-h,L-positions-h).min(axis=1))
        for k in np.argsort(-gap)[:count]:choices.append((float(gap[k]),np.r_[positions[k],angle]))
    choices.sort(key=lambda x:-x[0])
    return [np.vstack((p,q)) for _,q in choices[:count]]

def seeds(n,bank,rng,B):
    own=normalize(bank[n],0);out=[own.copy()];methods=['local squeeze']
    if n<1000:
        a=normalize(bank[n+1],0)
        h=(abs(np.cos(a[:,2]))+abs(np.sin(a[:,2])))/2
        d=cKDTree(a[:,:2]).query(a[:,:2],k=5)[0]
        ids=list(np.argsort(-d[:,1:].mean(1))[:8])
        for axis in (0,1):
            ids.extend(np.argsort(a[:,axis]-h)[:3]);ids.extend(np.argsort(-(a[:,axis]+h))[:3])
        for i in dict.fromkeys(ids):
            out.append(np.delete(a,i,0));methods.append(f'neighbour {n+1} remove {i} + CUDA squeeze')
    if n>2:
        p=normalize(bank[n-1],0)
        for a in insert_holes(p,rng):out.append(a);methods.append(f'neighbour {n-1} hole insertion + CUDA squeeze')
        # Also probe holes at centres occupied in the target's original topology.
        dist=cKDTree(p[:,:2]).query(own[:,:2])[0]
        for i in np.argsort(-dist)[:4]:
            out.append(np.vstack((p,own[i])));methods.append(f'neighbour {n-1} target-hole insertion + CUDA squeeze')
    while len(out)<B:
        parent=int(rng.integers(len(out)));p=out[parent].copy();label=methods[parent]
        if len(out)%4==0:
            # A small destroyed region permits a topology change during quench.
            count=int(rng.choice([2,3,5]));pivot=p[rng.integers(n),:2]
            ids=np.argsort(np.sum((p[:,:2]-pivot)**2,axis=1))[:count]
            p[ids,:2]+=rng.normal(0,.16,(count,2));p[ids,2]+=rng.normal(0,.12,count)
            label='regional basin hop from '+label
        else:
            scale=rng.choice([.0001,.001,.01,.03,.08])
            p+=rng.normal(0,scale,p.shape)*np.array([1,1,.3]);label='perturbed '+label
        out.append(p);methods.append(label)
    return np.stack([normalize(p,0) for p in out[:B]]),methods[:B]

def main():
    ap=argparse.ArgumentParser();ap.add_argument('--budget',type=float,default=6500);ap.add_argument('--per-n',type=float,default=45);ap.add_argument('--batch',type=int,default=48);ap.add_argument('--limit',type=int,default=200);ap.add_argument('--ns',type=int,nargs='*');ap.add_argument('--revisit',action='store_true');args=ap.parse_args()
    # Import torch first so its bundled runtime DLLs are available on Windows.
    import torch
    import cuda_backend as base
    import cuda_precision as precise
    base.check();precise.check()
    t0=time.time();dest=HERE/'gpu_candidates';dest.mkdir(exist_ok=True)
    donepath=HERE/'gpu_done.json';done=set(json.loads(donepath.read_text())) if donepath.exists() else set()
    bank={n:np.array(r,float) for n,r in load_submission(HERE.parent/'out_merge6/submission.csv').items()}
    original={n:side(p) for n,p in bank.items()}
    # Float incumbents can seed later search; official certification is a separate gate.
    for folder in ('cpu_candidates','gpu_candidates'):
        for path in (HERE/folder).glob('*.npz'):
            meta=json.loads(path.with_suffix('.json').read_text());n=int(meta['n']);p=np.load(path)['p']
            if side(p)<side(bank[n]):bank[n]=p
    table=pd.read_csv(HERE.parent/'out_merge6/submission_sources.csv')
    order=list(dict.fromkeys([n+d for n in PRIORITY for d in (-1,-2,-3,0,1,2,3)]+list(table.loc[table.source.str.contains('border|minus'),'n'])+list(range(2,1000))))
    if args.ns:order=args.ns
    order=[n for n in order if (args.revisit or n not in done) and n>2 and original[n]>np.sqrt(n)+1e-8][:args.limit]
    print(json.dumps(dict(event='start',budget=args.budget,jobs=len(order),batch=args.batch)),flush=True)
    for n in order:
        if (HERE/'STOP_GPU').exists():break
        control=HERE/'gpu_control.json'
        if control.exists():
            cfg=json.loads(control.read_text());args.per_n=float(cfg.get('per_n',args.per_n));args.budget=float(cfg.get('budget',args.budget))
        if time.time()-t0>=args.budget:break
        started=time.time();rng=np.random.default_rng(20261022+n);best=bank[n].copy();method='incumbent';roundno=0
        while time.time()-started<args.per_n and time.time()-t0<args.budget:
            if (HERE/'STOP_GPU').exists():break
            q,labels=seeds(n,bank,rng,args.batch)
            target=side(best)-rng.choice([1e-6,1e-5,1e-4,.001,.003],len(q))
            if roundno==0:target[:min(4,len(q))]=side(best)-1e-5
            for k in range(len(q)):
                L=side(q[k]);q[k,:,:2]=.5+(q[k,:,:2]-.5)*((target[k]-1)/(L-1))
            q,stats=precise.quench(q,target,1e-5,900)
            q,stats=precise.quench(q,target,1e-8,1400)
            q,stats=base.quench(q,target,1800)
            repaired=[]
            for i,p in enumerate(q):
                p=repair(p)
                if p is not None and side(p)<side(best)-2e-10:repaired.append((side(p),i,p))
            if repaired:
                repaired.sort(key=lambda t:t[0]);_,i,p=repaired[0]
                p,_,_=polish(p,rounds=60,seed=n+roundno,max_seconds=8)
                if side(p)<side(best):
                    best=p;method=labels[i];bank[n]=best
                    path=dest/f'gpu_{n}.npz';np.savez_compressed(path,p=best)
                    row=dict(n=n,method=method,old_L=original[n],new_L=side(best),gain=(original[n]-side(best))/np.sqrt(n),seconds=time.time()-started,round=roundno)
                    path.with_suffix('.json').write_text(json.dumps(row,indent=2));print(json.dumps(row),flush=True)
            roundno+=1
        done.add(n);donepath.write_text(json.dumps(sorted(done)))
        status=dict(n=n,method=method,before=original[n],after=side(best),seconds=time.time()-started,rounds=roundno,elapsed=time.time()-t0)
        print(json.dumps(status),flush=True)
        (HERE/'gpu_status.json').write_text(json.dumps(status,indent=2))
    print('COMPLETE',time.time()-t0,flush=True)

if __name__=='__main__':main()

In [ ]:
%%writefile search/fixed_lp.py
"""Fixed-angle sparse centre compaction; independent auxiliary experiment."""
import os
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='1'
import sys,time,json
from pathlib import Path
import numpy as np
from scipy.optimize import linprog
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from search_geometry import side,normalize,repair
HERE=Path(__file__).resolve().parent
sys.path.insert(0,str(HERE.parent))
from sq_merge import load_submission

def compact(p,seed=0,trust=.25):
    p=normalize(p,0);n=len(p);theta=p[:,2];L=side(p);rng=np.random.default_rng(seed)
    i,j=cKDTree(p[:,:2]).query_pairs(1.415+2*trust*np.sqrt(2),output_type='ndarray').T
    d=p[j,:2]-p[i,:2];c=np.cos(theta);s=np.sin(theta)
    ux=np.array([c[i],-s[i],c[j],-s[j]]).T;uy=np.array([s[i],c[i],s[j],c[j]]).T
    z=d[:,0,None]*ux+d[:,1,None]*uy;k=(abs(z)+rng.uniform(-1e-11,1e-11,z.shape)).argmax(1);rr=np.arange(len(i));sign=np.sign(z[rr,k])
    vx=sign*ux[rr,k];vy=sign*uy[rr,k]
    rad=(1+abs(np.cos(theta[i]-theta[j]))+abs(np.sin(theta[i]-theta[j])))/2+4e-11
    cc=np.array([2*i,2*i+1,2*j,2*j+1]).T
    vv=np.array([vx,vy,-vx,-vy]).T
    rows=list(np.repeat(np.arange(len(i)),4));cols=list(cc.ravel());vals=list(vv.ravel());rhs=list(-rad)
    h=(abs(c)+abs(s))/2
    for a in range(n):
        for dim in (0,1):
            row=len(rhs);rows.extend([row,row]);cols.extend([2*a+dim,2*n]);vals.extend([1.,-1.]);rhs.append(-h[a])
    A=coo_matrix((vals,(rows,cols)),shape=(len(rhs),2*n+1)).tocsr()
    obj=np.zeros(2*n+1);obj[-1]=1
    bounds=[(max(h[k],p[k,dim]-trust),p[k,dim]+trust) for k in range(n) for dim in (0,1)]+[(np.sqrt(n),L+2e-6)]
    fit=linprog(obj,A_ub=A,b_ub=rhs,bounds=bounds,method='highs',options={'primal_feasibility_tolerance':1e-10,'dual_feasibility_tolerance':1e-9,'time_limit':3})
    if not fit.success:return p
    q=p.copy();q[:,:2]=fit.x[:-1].reshape(n,2);q=repair(q)
    return q if q is not None and side(q)<L else p

def main():
    start=time.time();dest=HERE/'cpu_candidates';count=0;totalgain=0
    bank=load_submission(HERE.parent/'out_merge6/submission.csv')
    for n,rows in bank.items():
        p=np.array(rows,float)
        if side(p)<np.sqrt(n)+1e-8:continue
        q=compact(p,seed=n)
        if side(q)<side(p)-1e-9:
            row=dict(n=n,method='fixed-angle sparse LP compaction',before=side(p),after=side(q),gain=(side(p)-side(q))/np.sqrt(n))
            path=dest/f'fixed_{n}.npz';np.savez_compressed(path,p=q);path.with_suffix('.json').write_text(json.dumps(row,indent=2));print(json.dumps(row),flush=True)
            count+=1;totalgain+=row['gain']
    print(json.dumps(dict(event='COMPLETE',puzzles=len(bank),candidates=count,float_gain=totalgain,seconds=time.time()-start)),flush=True)

if __name__=='__main__':main()

In [ ]:
%%writefile search/fixed_transfer.py
"""Cheap fixed-angle LP screen of neighbour deletions."""
import os
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='1'
from pathlib import Path
import sys,json,time
from concurrent.futures import ProcessPoolExecutor,as_completed
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from fixed_lp import compact
from search_geometry import normalize,side
from run_cpu import PRIORITY
HERE=Path(__file__).resolve().parent
sys.path.insert(0,str(HERE.parent))
from sq_merge import load_submission

def worker(job):
    n,a,L=job;t=time.time();h=(abs(np.cos(a[:,2]))+abs(np.sin(a[:,2])))/2
    lo=(a[:,:2]-h[:,None]).min(0);hi=(a[:,:2]+h[:,None]).max(0)
    edge=np.minimum(a[:,:2]-h[:,None]-lo,hi-a[:,:2]-h[:,None]).min(1)
    ids=list(np.flatnonzero(edge<.02))
    d=cKDTree(a[:,:2]).query(a[:,:2],k=5)[0]
    ids=list(dict.fromkeys(list(np.argsort(-d[:,1:].mean(1))[:6])+ids))
    best=None;bestid=None;bestL=L
    for i in ids:
        q=compact(np.delete(a,i,0),seed=n+i,trust=.25)
        if side(q)<bestL-2e-10:best=q;bestL=side(q);bestid=int(i)
    row=dict(n=n,method=f'neighbour {n+1} remove {bestid} + fixed-angle LP',old_L=L,new_L=bestL,gain=(L-bestL)/np.sqrt(n),tested=len(ids),seconds=time.time()-t)
    return n,best,row

def main():
    bank={n:np.array(rows,float) for n,rows in load_submission(HERE.parent/'out_merge6/submission.csv').items()}
    table=pd.read_csv(HERE.parent/'out_merge6/submission_sources.csv')
    order=list(dict.fromkeys([n+d for n in PRIORITY for d in (-1,-2,-3,0,1,2,3)]+list(table.loc[table.source.str.contains('border|minus'),'n'])+list(range(2,1000))))
    jobs=[(n,bank[n+1],side(bank[n])) for n in order if 2<=n<1000 and side(bank[n])>np.sqrt(n)+1e-8 and side(bank[n+1])-side(bank[n])<.045]
    donepath=HERE/'fixed_transfer_done.json';done=set(json.loads(donepath.read_text())) if donepath.exists() else set()
    jobs=[job for job in jobs if job[0] not in done]
    print('start',len(jobs),flush=True)
    with ProcessPoolExecutor(max_workers=6) as pool:
        futures=[pool.submit(worker,job) for job in jobs]
        for f in as_completed(futures):
            n,p,row=f.result();print(json.dumps(row),flush=True)
            if p is not None:
                path=HERE/'cpu_candidates'/f'fixed_drop_{n}.npz';np.savez_compressed(path,p=p);path.with_suffix('.json').write_text(json.dumps(row,indent=2))
            done.add(n);donepath.write_text(json.dumps(sorted(done)))
    print('COMPLETE',flush=True)

if __name__=='__main__':main()

In [ ]:
%%writefile search/rotate_square_box.py
# Adapted from lightningv08, Apache-2.0.
"""Exact candidate enumeration for the smallest enclosing square of a rigid layout.

Whole-layout orientation polishing is used in Sas Pav's Santa-submission.
This independent implementation enumerates convex-hull support changes and
width/height crossings, rather than a single scalar local minimization.
"""
import json,math,time
from pathlib import Path
import numpy as np
from scipy.spatial import ConvexHull
from search_geometry import side,normalize
from search_geometry import repair

def rotate(p):
    c=np.cos(p[:,2])*.5;s=np.sin(p[:,2])*.5
    verts=np.concatenate([p[:,:2]+np.column_stack((sx*c-sy*s,sx*s+sy*c)) for sx,sy in [(1,1),(1,-1),(-1,1),(-1,-1)]])
    hull=verts[ConvexHull(verts).vertices];edges=np.roll(hull,-1,axis=0)-hull
    cuts=np.unique(np.r_[0.,np.mod(np.pi/2-np.arctan2(edges[:,1],edges[:,0]),np.pi/2),np.pi/2])
    candidates=list(cuts)
    for lo,hi in zip(cuts[:-1],cuts[1:]):
        mid=(lo+hi)/2;co,si=np.cos(mid),np.sin(mid);xx=hull[:,0]*co-hull[:,1]*si;yy=hull[:,0]*si+hull[:,1]*co
        dx=hull[xx.argmax()]-hull[xx.argmin()];dy=hull[yy.argmax()]-hull[yy.argmin()]
        A,B=dx[0],-dx[1];C,D=dy[1],dy[0]
        for a in [math.atan2(-(A-C),B-D),math.atan2(B,A),math.atan2(D,C)]:
            for k in [-1,0,1]:
                t=a+k*np.pi
                if lo<t<hi:candidates.append(t)
    aa=np.array(candidates);co=np.cos(aa);si=np.sin(aa)
    xx=hull[:,0,None]*co-hull[:,1,None]*si;yy=hull[:,0,None]*si+hull[:,1,None]*co
    sizes=np.maximum(np.ptp(xx,axis=0),np.ptp(yy,axis=0));a=float(aa[sizes.argmin()])
    if sizes.min()>side(p)-1e-9:return p,0.
    q=p.copy();co,si=np.cos(a),np.sin(a);q[:,:2]=p[:,:2]@np.array([[co,si],[-si,co]]);q[:,2]+=a
    q=repair(q)
    return (q,a) if q is not None and side(q)<side(p)-1e-9 else (p,0.)

def run(path='artifacts/gpu_public_portfolio_packings.npz',tag='global_rotation'):
    z=np.load(path);bank={int(n):z[n] for n in z.files};rows=[];at=time.perf_counter()
    for n,p in bank.items():
        q,a=rotate(p)
        if a:rows.append(dict(n=n,before=side(p),after=side(q),radians=a));bank[n]=q
    np.savez_compressed(f'artifacts/{tag}_packings.npz',**{str(n):p for n,p in bank.items()})
    r=dict(seconds=time.perf_counter()-at,rows=rows,score_gain=sum((r['before']-r['after'])/math.sqrt(r['n']) for r in rows),method='Convex-hull support intervals, interval endpoints, width/height crossings and stationary candidates')
    Path(f'research/{tag}_status.json').write_text(json.dumps(r,indent=2));print(json.dumps(r),flush=True)
    return bank,r

if __name__=='__main__':run()

In [ ]:
%%writefile search/refine_candidates.py
"""Additional contact-LP budget on the currently best floating candidates."""
import os
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='1'
from pathlib import Path
import sys,json,time
from concurrent.futures import ProcessPoolExecutor,as_completed
import numpy as np
from contact_polish import polish
from search_geometry import side
HERE=Path(__file__).resolve().parent

def worker(job):
    n,p,method=job;t=time.time()
    q,trace,_=polish(p,rounds=300,seed=20261022+n,max_seconds=50)
    return n,q,dict(n=n,method=method+'; extended contact LP',before=side(p),after=side(q),seconds=time.time()-t,steps=len(trace))

def main():
    bank={};methods={}
    receiptpath=HERE/'refine_receipts.json'
    receipts=json.loads(receiptpath.read_text()) if receiptpath.exists() else {}
    if not receipts and (HERE/'refine.log').exists():
        for line in (HERE/'refine.log').read_text().splitlines():
            try:
                row=json.loads(line)
                if 'after' in row:receipts[str(row['n'])]=row['after']
            except (ValueError,KeyError):pass
    for folder in ('cpu_candidates','gpu_candidates'):
        for path in (HERE/folder).glob('*.npz'):
            m=json.loads(path.with_suffix('.json').read_text());n=int(m['n']);p=np.load(path)['p']
            if n not in bank or side(p)<side(bank[n]):bank[n]=p;methods[n]=m['method']
    jobs=[(n,p,methods[n]) for n,p in bank.items() if str(n) not in receipts or side(p)<receipts[str(n)]-2e-10]
    print('start',len(jobs),'new or improved candidates;',len(bank),'total',flush=True)
    with ProcessPoolExecutor(max_workers=min(12,os.cpu_count() or 2)) as pool:
        futures=[pool.submit(worker,job) for job in jobs]
        for f in as_completed(futures):
            n,p,row=f.result();print(json.dumps(row),flush=True)
            if side(p)<side(bank[n])-2e-10:
                path=HERE/'cpu_candidates'/f'refined_{n}.npz';np.savez_compressed(path,p=p);path.with_suffix('.json').write_text(json.dumps(row,indent=2))
            receipts[str(n)]=side(p);receiptpath.write_text(json.dumps(receipts,indent=2))
    print('COMPLETE',flush=True)

if __name__=='__main__':main()

In [ ]:
%%writefile search/propagate.py
"""Propagate new search packings through the base library's free constructions."""
from pathlib import Path
import sys,json,time,math
import numpy as np
from search_geometry import side
HERE=Path(__file__).resolve().parent
sys.path.insert(0,str(HERE.parent))
from sq_merge import load_submission,float_rows,expand,drop_to

def main():
    t=time.time();bank={n:np.array(rows,float) for n,rows in load_submission(HERE.parent/'out_merge6/submission.csv').items()};improved=set();changes=[]
    for folder in ('cpu_candidates','gpu_candidates'):
        for path in (HERE/folder).glob('*.npz'):
            meta=json.loads(path.with_suffix('.json').read_text());n=int(meta['n']);p=np.load(path)['p']
            if side(p)<side(bank[n])-1e-11:bank[n]=p;improved.add(n)
    def save(n,q,method):
        row=dict(n=n,method=method,before=side(bank[n]),after=side(q),gain=(side(bank[n])-side(q))/np.sqrt(n))
        bank[n]=q;improved.add(n);path=HERE/'cpu_candidates'/f'propagate_{n}.npz';np.savez_compressed(path,p=q);path.with_suffix('.json').write_text(json.dumps(row,indent=2));changes.append(row);print(json.dumps(row),flush=True)
    for n in range(2,1001):
        if n not in improved:continue
        p=bank[n];L=side(p);f=math.floor(L-1e-9)
        for j in range(1,7):
            m=n+j*(2*f+j)
            if m>1000:break
            if L+j>=side(bank[m])-1e-9:continue
            rows=expand(float_rows(p,dilate=0),j);q=np.array(rows,float)
            assert len(q)==m
            if side(q)<side(bank[m])-1e-11:save(m,q,f'border +{j} from optimized n={n}')
    source=1000
    for n in range(999,1,-1):
        if side(bank[n])<side(bank[source]):source=n
        elif source in improved and side(bank[source])<side(bank[n])-1e-9:
            q=np.array(drop_to(float_rows(bank[source],dilate=0),n),float)
            save(n,q,f'deletion from optimized n={source}')
    print('COMPLETE',json.dumps(dict(changes=len(changes),seconds=time.time()-t)),flush=True)

if __name__=='__main__':main()

In [ ]:
if RECOMPUTE_SEARCH:
    Path('out_merge6').mkdir(exist_ok=True)
    write_submission(base,'out_merge6/submission.csv')
    sources=base_path.with_name('submission_sources.csv')
    assert sources.exists(),sources
    shutil.copyfile(sources,'out_merge6/submission_sources.csv')
    subprocess.run([sys.executable,'-u','search/run_cpu.py','--mode','own','--limit','260'],check=True)
    subprocess.run([sys.executable,'-u','search/fixed_lp.py'],check=True)
    subprocess.run([sys.executable,'-u','search/fixed_transfer.py'],check=True)
    # Requires a GPU runtime and CuPy; at most 6500 seconds, below two hours.
    subprocess.run([sys.executable,'-u','search/run_gpu.py','--budget','6500'],check=True)
    subprocess.run([sys.executable,'-u','search/propagate.py'],check=True)
    subprocess.run([sys.executable,'-u','search/refine_candidates.py'],check=True)

## Replay, certify, and write

Every candidate is checked on the exact decimal strings that will be written. The complete 1000-puzzle output is certified again.

In [ ]:
REPLAY_B64='...'
REPLAY_NS=[268, 303, 338, 372, 376, 416, 417, 418, 423, 456, 457, 458, 460, 461, 465, 495, 500, 504, 505, 506, 509, 542, 544, 546, 548, 550, 551, 552, 555, 598, 600, 601, 602, 639, 649, 651, 652, 653, 693, 703, 750, 757, 758, 759, 801, 804, 814, 816, 858, 873, 918, 919, 920, 929, 934, 983, 993]
EXPECTED_TOTAL=22.307808566601995
METHODS={'416': 'CPU contact LP (own_416)', '750': 'CPU contact LP (own_750)', '544': 'CPU contact LP (own_544)', '649': 'CPU contact LP (own_649)', '919': 'CPU contact LP (own_919)', '303': 'CPU contact LP (own_303)', '550': 'CPU contact LP (own_550)', '458': 'CPU contact LP (own_458)', '757': 'CPU contact LP (own_757)', '858': 'CPU contact LP (own_858)', '509': 'CPU contact LP (own_509)', '500': 'CPU contact LP (own_500)', '495': 'CPU contact LP (own_495)', '460': 'CPU contact LP (own_460)', '551': 'CPU contact LP (own_551)', '461': 'CPU contact LP (own_461)', '920': 'CPU contact LP (own_920)', '602': 'CPU contact LP (own_602)', '983': 'CPU contact LP (own_983)', '542': 'CPU contact LP (own_542)', '504': 'CPU contact LP (own_504)', '703': 'CPU contact LP (own_703)', '506': 'CPU contact LP (own_506)', '651': 'CPU contact LP (own_651)', '816': 'CPU contact LP (own_816)', '268': 'CPU contact LP (own_268)', '457': 'CPU contact LP (own_457)', '555': 'CPU contact LP (own_555)', '552': 'CPU contact LP (own_552)', '801': 'CPU contact LP (own_801)', '423': 'CPU contact LP (own_423)', '338': 'CPU contact LP (own_338)', '873': 'CPU contact LP (own_873)', '418': 'CPU contact LP (own_418)', '456': 'CPU contact LP (own_456)', '653': 'CPU contact LP (own_653)', '598': 'CPU contact LP (own_598)', '546': 'CPU contact LP (own_546)', '600': 'CPU contact LP (own_600)', '465': 'CPU contact LP (own_465)', '693': 'CPU contact LP (own_693)', '548': 'CPU contact LP (own_548)', '804': 'CPU contact LP (own_804)', '993': 'CPU contact LP (own_993)', '758': 'CPU contact LP (own_758)', '918': 'CPU contact LP (own_918)', '376': 'CPU contact LP (own_376)', '759': 'CPU contact LP (own_759)', '934': 'CPU contact LP (own_934)', '639': 'CPU contact LP (own_639)', '372': 'CPU contact LP (own_372)', '814': 'CPU contact LP (own_814)', '652': 'CPU contact LP (own_652)', '601': 'CPU contact LP (own_601)', '505': 'CPU contact LP (own_505)', '417': 'CPU contact LP (own_417)', '929': 'CPU contact LP (own_929)'}

In [ ]:
matrix=np.frombuffer(lzma.decompress(base64.b64decode(REPLAY_B64)),dtype='<f8').reshape(3,-1).T
replay={};offset=0
for n in REPLAY_NS:
    replay[n]=[tuple(repr(float(v)) for v in row) for row in matrix[offset:offset+n]]
    offset+=n
assert offset==len(matrix)
if RECOMPUTE_SEARCH:
    for folder in ('cpu_candidates','gpu_candidates'):
        for path in Path('search',folder).glob('*.npz'):
            meta=json.loads(path.with_suffix('.json').read_text());n=int(meta['n'])
            rows=float_rows(np.load(path)['p'],dilate=1e-11)
            if float_side(rows)<float_side(replay.get(n,base[n])):
                replay[n]=rows;METHODS[str(n)]=meta['method']
checks=certify(replay,workers=WORKERS)
solution=base.copy();gains=[]
for n in sorted(replay):
    score,L,reason=checks[n]
    assert reason=='ok',(n,reason)
    if score<base_checks[n][0]:
        solution[n]=replay[n]
        gains.append(dict(n=n,old_L=float(base_checks[n][1]),new_L=float(L),method=METHODS[str(n)],gain=base_checks[n][0]-score))
write_submission(solution,'submission.csv')
final_checks=certify(load_submission('submission.csv'),workers=WORKERS)
assert len(final_checks)==1000 and all(r=='ok' for s,L,r in final_checks.values())
gain_table=pd.DataFrame(gains)
gain_table.to_csv('improvements.csv',index=False)
assert len(pd.read_csv('submission.csv'))==500500
print(gain_table.to_string(index=False))

## Merge with the newest public packings

Public notebooks move fast. Each attached public output is compared puzzle by puzzle with the packing above.
A smaller public packing replaces ours only if it passes the official metric after the same float64 round trip the scorer uses.
If the best one fails, the next best is tried.

In [ ]:
%%writefile merge_public.py
"""Per-n: keep our packing unless a newer public packing is smaller AND certifies after the float64 round trip."""
import sys, glob, os, time
import pandas as pd, numpy as np
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from sq_merge import load_submission, float_side, as_scored, certify, write_submission

def merge(base_path, pub_paths, out_path, workers=4, log=print):
    t0 = time.time()
    sol = {n: as_scored(r) for n, r in load_submission(base_path).items()}
    cur = {n: float_side(r) for n, r in sol.items()}
    pubs = {p: load_submission(p) for p in pub_paths}
    src = {n: 'base' for n in sol}
    # candidates per n, best first; try in rounds until each n certifies or runs out
    cands = {}
    for p, sub in pubs.items():
        for n, rows in sub.items():
            L = float_side(rows)
            if L < cur[n] - 1e-12:
                cands.setdefault(n, []).append((L, p))
    for n in cands:
        cands[n].sort()
    log(f'{len(cands)} puzzles have a smaller public packing ({time.time() - t0:.0f}s)')
    pending = dict(cands)
    while pending:
        trial = {n: as_scored(pubs[c[0][1]][n]) for n, c in pending.items()}
        res = certify(trial, workers)
        nxt = {}
        for n, (s, L, r) in res.items():
            if r == 'ok':
                sol[n] = trial[n]; src[n] = pending[n][0][1]
            elif len(pending[n]) > 1:
                nxt[n] = pending[n][1:]
        log(f'round: {sum(r == "ok" for _, _, r in res.values())} accepted, {len(nxt)} retry')
        pending = nxt
    write_submission(sol, out_path)
    return src

if __name__ == '__main__':
    base, out = sys.argv[1], sys.argv[2]
    pubs = sys.argv[3:]
    src = merge(base, pubs, out, workers=16)
    print(pd.Series(src).map(lambda s: s.split(os.sep)[-2] if os.sep in s or '/' in s else s).value_counts().to_string())

In [ ]:
from merge_public import merge
ours_total=sum(final_checks[n][0] for n in sorted(final_checks))
public=[p for p in paths if p!=base_path]  # a fork of this notebook may be a legitimate input, so no name-based exclusion
print('public inputs:',*map(str,public),sep='\n  ')
shutil.copyfile('submission.csv','submission_ours.csv')
merge_src=merge('submission_ours.csv',public,'submission.csv',workers=WORKERS)
final_checks=certify(load_submission('submission.csv'),workers=WORKERS)
assert len(final_checks)==1000 and all(r=='ok' for s,L,r in final_checks.values())
merged_total=sum(final_checks[n][0] for n in sorted(final_checks))
kept=sum(v=='base' for v in merge_src.values())
print(f'ours: {ours_total:.9f}   after public merge: {merged_total:.9f}   puzzles kept from this notebook: {kept}')
# free constructions on the merged packings: border expansion and deletion (no new certificates needed)
from sq_merge import merge_all
Path('no_certs').mkdir(exist_ok=True)
shutil.copyfile('submission.csv','submission_merged.csv')
table=merge_all(['submission_merged.csv'],'no_certs',str(ROOT),workers=WORKERS)
final_checks=certify(load_submission('submission.csv'),workers=WORKERS)
assert len(final_checks)==1000 and all(r=='ok' for s,L,r in final_checks.values())
print(f'after border expansion / deletion: {sum(final_checks[n][0] for n in sorted(final_checks)):.9f}')

## Score the final float64 file

Default `pd.read_csv` followed by the official `score()` reproduces the leaderboard read path.

In [ ]:
import metric
metric.KAGGLE_CORES=WORKERS
t=time.time()
total=metric.score(pd.DataFrame({'id':range(1000),'n':range(1,1001)}),pd.read_csv('submission.csv'),'id')
exact_total=sum(final_checks[n][0] for n in sorted(final_checks))
assert abs(total-exact_total)<1e-10,(total,exact_total)
result=dict(before=before,ours=ours_total,after=total,gain=before-total,improved=len(gains),all_certified=True,rows=500500,score_seconds=time.time()-t)
Path('validation.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))

## Measured gains and improved packings

In [ ]:
# compare with the best single attached notebook (lowest total score)
def total_score(sub):
    return sum((float_side(sub[n]) if n in sub else np.ceil(np.sqrt(n)))/np.sqrt(n)-1 for n in range(1,1001))
subs_pub={p:load_submission(p) for p in public}
best_pub=min(subs_pub,key=lambda p:total_score(subs_pub[p]))
ref=subs_pub[best_pub]
final=load_submission('submission.csv')
gain_table=pd.DataFrame([dict(n=n,gain=((float_side(ref[n]) if n in ref else np.ceil(np.sqrt(n)))-float_side(final[n]))/np.sqrt(n)) for n in range(1,1001)])
gain_table=gain_table[gain_table.gain>1e-12]
solution=final
print(f'best single attached notebook: {best_pub} ({total_score(ref):.7f})')
print(f'gain over it: {gain_table.gain.sum():.3e} across {len(gain_table)} puzzles')
fig,ax=plt.subplots(figsize=(11,3.5))
ordered= gain_table.sort_values('gain',ascending=False)
visible=ordered.head(40)
ax.bar(visible.n.astype(str),visible.gain,color='#147d92')
label=' (largest 40 shown)' if len(ordered)>40 else ''
ax.set(xlabel='Puzzle n',ylabel='Reduction in total score',title=f"Gain over best single notebook: {gain_table.gain.sum():.2e} across {len(gain_table)} puzzles{label}")
ax.tick_params(axis='x',rotation=70);fig.tight_layout();fig.savefig('gains.png',dpi=160);plt.show()
show=ordered.n.head(4).tolist() or [int(n) for n in REPLAY_NS[:4]]
fig,axes=plt.subplots(1,len(show),figsize=(5*len(show),5.4),squeeze=False)
corners=np.array([[-.5,-.5],[.5,-.5],[.5,.5],[-.5,.5]])
for ax,n in zip(axes[0],show):
    p=np.array(solution[n],float)
    for x,y,a in p:
        c,s=np.cos(a),np.sin(a)
        q=corners@np.array([[c,s],[-s,c]])+[x,y]
        ax.add_patch(Polygon(q,facecolor=plt.cm.viridis(abs(np.sin(2*a))),edgecolor='white',linewidth=.25))
    h=(abs(np.cos(p[:,2]))+abs(np.sin(p[:,2])))/2
    lo=(p[:,:2]-h[:,None]).min(0);L=float(final_checks[n][1])
    ax.set_xlim(lo[0]-.1,lo[0]+L+.1);ax.set_ylim(lo[1]-.1,lo[1]+L+.1)
    ax.set_aspect('equal');ax.set_title(f'n={n}, L={L:.9f}',fontsize=11,pad=12);ax.axis('off')
fig.subplots_adjust(left=.01,right=.99,bottom=.02,top=.86,wspace=.08)
fig.savefig('improved_packings.png',dpi=180);plt.show()

## Credits and limitations

* Base packings and exact literature witnesses: [Fresh Records, Certified](https://www.kaggle.com/code/malyshevdanil/squares-in-squares-fresh-records-certified), its attached public notebooks, Francisco Couzo, Evan Daniel, Joshua Levy and all original contributors credited there. Derived data retain those credits.
* CUDA spatial hashing, per-island L-BFGS, conservative SAT continuation, and sparse contact LP: [lightningv08 — CUDA Packing Search](https://www.kaggle.com/code/lightningv08/squares-in-squares-cuda-packing-search), Apache-2.0. The copied source and its background credit to prvsiyan are retained.
* Related window-closure and contact-polish approach: [kirilltym — CPU Window Closure + Contact Polish](https://www.kaggle.com/code/kirilltym/22-590879-cpu-window-closure-contact-polish). Its executable notebook cells were not run during extraction.
* Public packings merged in version 78: [alexchilton — fault line dp basin hopping lb 22 2620](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2620), [alexchilton — fault line dp basin hopping lb 22 2672](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2672), [alexchilton — fault line dp basin hopping lb 22 2990](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2990), [alexchilton — squares 101 the starter kit 2 600 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-2-600-downloads), [alexchilton — squares 101 the starter kit 3 700 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-3-700-downloads), [alexchilton — squares 101 the starter kit 4 400 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-4-400-downloads), [alexchilton — squares 101 the starter kit 4 900 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-4-900-downloads), [alexchilton — squares 101 the starter kit 5 800 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-5-800-downloads), [alexchilton — squares 101 the starter kit 6 200 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-6-200-downloads), [alexchilton — squares 101 the starter kit 6 700 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-6-700-downloads), [alexchilton — squares 101 the starter kit 6 800 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-6-800-downloads), [alexchilton — squares 101 the starter kit 7 100 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-7-100-downloads), [alexchilton — squares 101 the starter kit 7 200 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-7-200-downloads), [alexchilton — squares 101 the starter kit 7 300 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-7-300-downloads), [alexchilton — squares 101 the starter kit 7 500 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-7-500-downloads), [alexchilton — squares 101 the starter kit 750 downloads](https://www.kaggle.com/code/alexchilton/squares-101-the-starter-kit-750-downloads), [alimaadl3lim3d — squares in squares exact contact fix lb 22 26129](https://www.kaggle.com/code/alimaadl3lim3d/squares-in-squares-exact-contact-fix-lb-22-26129), [aqibrazadev — squares in squares top lb blending visualization](https://www.kaggle.com/code/aqibrazadev/squares-in-squares-top-lb-blending-visualization), [areebbb — squares in squares submission](https://www.kaggle.com/code/areebbb/squares-in-squares-submission), [avazaku — c seam dp certified merge](https://www.kaggle.com/code/avazaku/c-seam-dp-certified-merge), [avazaku — certified merge cuda rectangular cycles](https://www.kaggle.com/code/avazaku/certified-merge-cuda-rectangular-cycles), [avazaku — squares c contact modes sat lp](https://www.kaggle.com/code/avazaku/squares-c-contact-modes-sat-lp), [avazaku — squares in squares c contact modes sat lp](https://www.kaggle.com/code/avazaku/squares-in-squares-c-contact-modes-sat-lp), [avazaku — squares in squares c sat lp](https://www.kaggle.com/code/avazaku/squares-in-squares-c-sat-lp), [az05192000gmailcom — l band builds re quench and the 1 packings](https://www.kaggle.com/code/az05192000gmailcom/l-band-builds-re-quench-and-the-1-packings), [azharalisoomro — squares 101 the starter kit 3 700 downloads](https://www.kaggle.com/code/azharalisoomro/squares-101-the-starter-kit-3-700-downloads), [busyaprime — 33 lb 22 2630316 public packings fq quench](https://www.kaggle.com/code/busyaprime/33-lb-22-2630316-public-packings-fq-quench), [busyaprime — 33 lb 22 2683156 public packings fq quench](https://www.kaggle.com/code/busyaprime/33-lb-22-2683156-public-packings-fq-quench), [busyaprime — lb 22 2546533 best public packings fq hop](https://www.kaggle.com/code/busyaprime/lb-22-2546533-best-public-packings-fq-hop), [busyaprime — lb 22 2569679 best public packings fq hop](https://www.kaggle.com/code/busyaprime/lb-22-2569679-best-public-packings-fq-hop), [busyaprime — lb 22 2569880 best public packings fq hop](https://www.kaggle.com/code/busyaprime/lb-22-2569880-best-public-packings-fq-hop), [busyaprime — lb 22 2591302 best public packings fq hop](https://www.kaggle.com/code/busyaprime/lb-22-2591302-best-public-packings-fq-hop), [busyaprime — lb 22 2593018 best public packings as doubles](https://www.kaggle.com/code/busyaprime/lb-22-2593018-best-public-packings-as-doubles), [busyaprime — lb 22 2616222 best public packings fq quench](https://www.kaggle.com/code/busyaprime/lb-22-2616222-best-public-packings-fq-quench), [busyaprime — lb 22 2617195 best public packings fq quench](https://www.kaggle.com/code/busyaprime/lb-22-2617195-best-public-packings-fq-quench), [busyaprime — lb 22 2619999 best public packings fq quench](https://www.kaggle.com/code/busyaprime/lb-22-2619999-best-public-packings-fq-quench), [busyaprime — lb 22 2622907 best public packings fq quench](https://www.kaggle.com/code/busyaprime/lb-22-2622907-best-public-packings-fq-quench), [busyaprime — lb 22 2623982 best public packings fq quench](https://www.kaggle.com/code/busyaprime/lb-22-2623982-best-public-packings-fq-quench), [evgendvorkin — squares in squares deep eda analogies](https://www.kaggle.com/code/evgendvorkin/squares-in-squares-deep-eda-analogies), [jazivxt — boxed in sub](https://www.kaggle.com/code/jazivxt/boxed-in-sub), [kirilltym — 22 2499 cpu cuda mps eda budget lp](https://www.kaggle.com/code/kirilltym/22-2499-cpu-cuda-mps-eda-budget-lp), [kirilltym — 22 2502 cpu cuda mps eda portfolio lp](https://www.kaggle.com/code/kirilltym/22-2502-cpu-cuda-mps-eda-portfolio-lp), [kirilltym — 22 2502 cpu cuda mps eda trust lp](https://www.kaggle.com/code/kirilltym/22-2502-cpu-cuda-mps-eda-trust-lp), [kirilltym — 22 2509 cpu cuda mps eda continue lp](https://www.kaggle.com/code/kirilltym/22-2509-cpu-cuda-mps-eda-continue-lp), [kirilltym — 22 2510 cpu cuda mps eda round lp](https://www.kaggle.com/code/kirilltym/22-2510-cpu-cuda-mps-eda-round-lp), [kirilltym — 22 2540 cpu cuda mps eda contact map](https://www.kaggle.com/code/kirilltym/22-2540-cpu-cuda-mps-eda-contact-map), [kirilltym — 22 2540 cpu cuda mps eda revisit lp](https://www.kaggle.com/code/kirilltym/22-2540-cpu-cuda-mps-eda-revisit-lp), [kirilltym — 22 2541 cpu cuda mps eda angle lp](https://www.kaggle.com/code/kirilltym/22-2541-cpu-cuda-mps-eda-angle-lp), [kirilltym — 22 2542 cpu cuda mps eda axis study](https://www.kaggle.com/code/kirilltym/22-2542-cpu-cuda-mps-eda-axis-study), [kirilltym — 22 2546 cpu cuda mps eda axis lp](https://www.kaggle.com/code/kirilltym/22-2546-cpu-cuda-mps-eda-axis-lp), [kirilltym — 22 2546 cpu cuda mps eda complement lp](https://www.kaggle.com/code/kirilltym/22-2546-cpu-cuda-mps-eda-complement-lp), [kirilltym — 22 2546 cpu cuda mps eda margin lp](https://www.kaggle.com/code/kirilltym/22-2546-cpu-cuda-mps-eda-margin-lp), [kirilltym — 22 2546 cpu cuda mps eda witness](https://www.kaggle.com/code/kirilltym/22-2546-cpu-cuda-mps-eda-witness), [kirilltym — 22 2558 cpu cuda mps eda contact repair](https://www.kaggle.com/code/kirilltym/22-2558-cpu-cuda-mps-eda-contact-repair), [kirilltym — square packing lab cpu cuda mps](https://www.kaggle.com/code/kirilltym/square-packing-lab-cpu-cuda-mps), [kirilltym — squares in squares cpu cuda mps eda projunion](https://www.kaggle.com/code/kirilltym/squares-in-squares-cpu-cuda-mps-eda-projunion), [kmmmvs — squares dashboard public batch 20261027](https://www.kaggle.com/code/kmmmvs/squares-dashboard-public-batch-20261027), [krishnayadav456wrsty — learn with me squares in squares 22 2619730280](https://www.kaggle.com/code/krishnayadav456wrsty/learn-with-me-squares-in-squares-22-2619730280), [krishnayadav456wrsty — learn with me v1 squares in squares 22 2619730280](https://www.kaggle.com/code/krishnayadav456wrsty/learn-with-me-v1-squares-in-squares-22-2619730280), [lianlian99 — 11 3 notebook](https://www.kaggle.com/code/lianlian99/11-3-notebook), [lianlian99 — notebook0dc062f649](https://www.kaggle.com/code/lianlian99/notebook0dc062f649), [lucifer19 — think select reclose](https://www.kaggle.com/code/lucifer19/think-select-reclose), [lucifer19 — think select reclose v24 the frontier moves](https://www.kaggle.com/code/lucifer19/think-select-reclose-v24-the-frontier-moves), [mdmuqtadirfuad — squares in squares certified 22 2563 method](https://www.kaggle.com/code/mdmuqtadirfuad/squares-in-squares-certified-22-2563-method), [mdmuqtadirfuad — squares in squares certified method](https://www.kaggle.com/code/mdmuqtadirfuad/squares-in-squares-certified-method), [mufasasimon — squares in squares float64 trap gap map merge](https://www.kaggle.com/code/mufasasimon/squares-in-squares-float64-trap-gap-map-merge), [navneetprabhat — squares in squares compilation by instinct](https://www.kaggle.com/code/navneetprabhat/squares-in-squares-compilation-by-instinct), [nilanjansaha123 — certified merge l band quench engine](https://www.kaggle.com/code/nilanjansaha123/certified-merge-l-band-quench-engine), [nilanjansaha123 — certified merge projection quench gpu screen](https://www.kaggle.com/code/nilanjansaha123/certified-merge-projection-quench-gpu-screen), [nilanjansaha123 — hybrid slp basin hopper t4x2](https://www.kaggle.com/code/nilanjansaha123/hybrid-slp-basin-hopper-t4x2), [nilanjansaha123 — s in s certified portfolio merge proj quench](https://www.kaggle.com/code/nilanjansaha123/s-in-s-certified-portfolio-merge-proj-quench), [nilanjansaha123 — s in s certified portfolio merge proj quench2](https://www.kaggle.com/code/nilanjansaha123/s-in-s-certified-portfolio-merge-proj-quench2), [nilanjansaha123 — s in s seam dp basin hop kink lp quench](https://www.kaggle.com/code/nilanjansaha123/s-in-s-seam-dp-basin-hop-kink-lp-quench), [nilanjansaha123 — seam dp multi basin hop kink lp quench](https://www.kaggle.com/code/nilanjansaha123/seam-dp-multi-basin-hop-kink-lp-quench), [ntumlnoob — 22 264 with rectangles ftmc milp fq cpu gpu](https://www.kaggle.com/code/ntumlnoob/22-264-with-rectangles-ftmc-milp-fq-cpu-gpu), [ntumlnoob — 22 2640194438 with rectangles ftmc milp fq cpu gpu](https://www.kaggle.com/code/ntumlnoob/22-2640194438-with-rectangles-ftmc-milp-fq-cpu-gpu), [ntumlnoob — first 22 266024737612 public code with explanation](https://www.kaggle.com/code/ntumlnoob/first-22-266024737612-public-code-with-explanation), [ntumlnoob — first 22 268681355 public code with explanation](https://www.kaggle.com/code/ntumlnoob/first-22-268681355-public-code-with-explanation), [octaviograu — squares certified portfolio blend](https://www.kaggle.com/code/octaviograu/squares-certified-portfolio-blend), [pramonoutomo — sis best packings 22 2498853780 kaggle verified](https://www.kaggle.com/code/pramonoutomo/sis-best-packings-22-2498853780-kaggle-verified), [prvsiyan — a square deal research to verified packings](https://www.kaggle.com/code/prvsiyan/a-square-deal-research-to-verified-packings), [prvsiyan — auto updater](https://www.kaggle.com/code/prvsiyan/auto-updater), [ridhamkavani — squares in squares neighbour transfer squeeze](https://www.kaggle.com/code/ridhamkavani/squares-in-squares-neighbour-transfer-squeeze), [saamhm — auto optimizer packing unit squares](https://www.kaggle.com/code/saamhm/auto-optimizer-packing-unit-squares), [saamhm — optimization until convergence](https://www.kaggle.com/code/saamhm/optimization-until-convergence), [seshurajup — 1000 puzzles ensemble lb 22 2540](https://www.kaggle.com/code/seshurajup/1000-puzzles-ensemble-lb-22-2540), [seshurajup — 1000 puzzles ensemble lb 22 298819](https://www.kaggle.com/code/seshurajup/1000-puzzles-ensemble-lb-22-298819), [sigmaborov — squares in squares certified contact polish](https://www.kaggle.com/code/sigmaborov/squares-in-squares-certified-contact-polish), [pjmathematician — Three square packings that beat the records](https://www.kaggle.com/code/pjmathematician/three-square-packings-that-beat-the-records) (packings for n = 299, 260 and 232, embedded and re-certified here), with every source and credit they list.
* Official competition metric: Apache-2.0, embedded unchanged.

The optimizer is heuristic. Many starts remain jammed or fail to beat the base; floating geometry alone is never accepted. Small gains should not be interpreted as a proof of optimality. This notebook only writes `submission.csv`; it does not submit anything itself.